# 🌲 Calibration Tree - Precision Resistances

## 🎯 Objective

Cross-calibration of **4 precision resistance sets** through offset chaining.

## 📊 Structure

- **Temperature references (not used as raised)**: Sensors 1009 and 1010 (channels 13 and 14)
  - Used internally within each set for offset calculation
  - NOT used to chain offsets between different sets
- **First round (R1)**: 4 resistance sets
  - RESIST_SET1: PDHD-HP-13 to PDHD-HP-24
  - RESIST_SET2: PDHD-HP-25 to PDHD-HP-36
  - RESIST_SET3: PDHD-HP-37 to PDHD-HP-48
  - RESIST_SET4: PDHD-HP-49 to PDHD-HP-60
- **Second round (R2)**: 1 set with 3 resistances from each R1 set
  - RESIST_SET5: Mix of 12 resistances (3 from each R1 set)

## ⚙️ Method

1. **Per-set processing**: Calculate offsets and constants within each set
2. **Identify "raised sensors"**: Resistance sensors that appear in both R1 and R2
3. **Offset chaining**: Connect R1 resistances through R2 using raised sensors as bridges
4. **Final calibration constants**: Calculate offsets between any two resistances
   - Same set: Direct offset calculation (using internal reference)
   - Different sets: Multi-path calculation via R2 raised sensors
   - Weighted average: Weight = 1/error² (gives more importance to precise paths)

## 🔧 Classes used

- `SetSTS`: Adapted version for resistances
- `RunSTS`: Individual run handling
- `CalibrationNetwork`: Calibration graph construction (optional)

In [1]:
# =============================================================================
# SETUP E IMPORTS
# =============================================================================

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

# Add paths
project_path = os.path.abspath("../../")
sys.path.append(project_path)
src_dir = os.path.abspath("../src")
sys.path.append(src_dir)

print("🔍 Paths configured:")
print(f"  - Project path: {project_path}")
print(f"  - Src dir: {src_dir}")

# Imports
try:
    import importlib
    if 'RTD_Calibration_VGP.src.calibration_network' in sys.modules:
        importlib.reload(sys.modules['RTD_Calibration_VGP.src.calibration_network'])
    
    from RTD_Calibration_VGP.src.calibration_network import CalibrationNetwork
    from RTD_Calibration_VGP.src.setSTS import SetSTS
    from RTD_Calibration_VGP.src.logfile import Logfile
    print("✅ Imports from RTD_Calibration_VGP.src completed")
except ImportError as e:
    print(f"⚠️ Error importing from RTD_Calibration_VGP.src: {e}")
    try:
        from calibration_network import CalibrationNetwork
        from setSTS import SetSTS
        from logfile import Logfile
        print("✅ Local imports completed")
    except ImportError as e2:
        print(f"❌ Error importing classes: {e2}")
        raise e2

print("✅ Setup completed")
print("📁 Working directory:", os.getcwd())

🔍 Paths configured:
  - Project path: /Users/vicky/Desktop/rtd-calibration-ana
  - Src dir: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/src
✅ Imports from RTD_Calibration_VGP.src completed
✅ Setup completed
📁 Working directory: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/notebooks


## 📂 Data Loading

In [2]:
# ------------------------------------------------------------------
# LOAD LOGFILE
# ------------------------------------------------------------------
print("="*80)
print("📂 LOADING LOGFILE")
print("="*80)

logfile_paths = [
    "../data/LogFile.csv",
    "RTD_Calibration_VGP/data/LogFile.csv",
    "../../data/LogFile.csv"
]

logfile = None
for path in logfile_paths:
    if os.path.exists(path):
        try:
            logfile = Logfile(path)
            print(f"✅ Logfile loaded from {path}")
            print(f"   Total records: {len(logfile.log_file)}")
            break
        except Exception as e:
            print(f"⚠️ Error: {e}")

if logfile is None:
    raise FileNotFoundError("Could not find the logfile")

# Filter only RESIST_SET
resist_data = logfile.log_file[
    logfile.log_file['CalibSetNumber'].astype(str).str.contains('RESIST_SET', na=False)
]
print(f"\n📊 RESIST_SET records: {len(resist_data)}")
print(f"   Sets found: {sorted(resist_data['CalibSetNumber'].unique())}")

📂 LOADING LOGFILE
CSV file loaded successfully from '../data/LogFile.csv'.
✅ Logfile loaded from ../data/LogFile.csv
   Total records: 832

📊 RESIST_SET records: 21
   Sets found: ['RESIST_SET0', 'RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4', 'RESIST_SET5']


## ⚙️ Round Configuration

We manually define the rounds since we have a simple tree structure:
- **Round 1**: RESIST_SET1, RESIST_SET2, RESIST_SET3, RESIST_SET4
- **Round 2**: RESIST_SET5 (contains 3 resistances from each R1 set)

In [3]:
# ------------------------------------------------------------------
# ROUND CONFIGURATION
# ------------------------------------------------------------------

# Manual mapping of sets to rounds
sets_config = {
    'RESIST_SET1': {'round': 1, 'description': 'PDHD-HP-13 to PDHD-HP-24'},
    'RESIST_SET2': {'round': 1, 'description': 'PDHD-HP-25 to PDHD-HP-36'},
    'RESIST_SET3': {'round': 1, 'description': 'PDHD-HP-37 to PDHD-HP-48'},
    'RESIST_SET4': {'round': 1, 'description': 'PDHD-HP-49 to PDHD-HP-60'},
    'RESIST_SET5': {'round': 2, 'description': 'Mix: 3 from each R1 set'}
}

# NOTE: SetSTS uses channel 2 as internal reference to calculate offsets within each set
# The sensor in channel 2 varies depending on the set.
# 
# Temperature sensors 1009 and 1010 are present in channels 13 and 14,
# and are NOT used to chain offsets between different sets.
# 
# Chaining R1 → R2 is done through RAISED SENSORS
# (resistance sensors that appear in both R1 and R2)

print("⚙️ CONFIGURATION:")
print("   📌 Internal reference: Channel 2 (varies per set)")
print("   🔗 Offset chaining: Via RAISED resistance sensors (R1 ∩ R2)")
print("   ⚠️  Temperature sensors 1009/1010 (ch 13-14) are NOT used for chaining")
print("\n   Sets per round:")
for ronda in [1, 2]:
    sets_in_round = [s for s, cfg in sets_config.items() if cfg['round'] == ronda]
    print(f"   - Round {ronda}: {sets_in_round}")

⚙️ CONFIGURATION:
   📌 Internal reference: Channel 2 (varies per set)
   🔗 Offset chaining: Via RAISED resistance sensors (R1 ∩ R2)
   ⚠️  Temperature sensors 1009/1010 (ch 13-14) are NOT used for chaining

   Sets per round:
   - Round 1: ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']
   - Round 2: ['RESIST_SET5']


## 🔄 Set Processing

In [4]:
# ------------------------------------------------------------------
# PROCESS ALL RESIST_SET
# ------------------------------------------------------------------
print("="*80)
print("🔄 SET PROCESSING")
print("="*80)

processed_sets = {}
all_sensors = set()

# Create a single SetSTS instance for all RESIST_SET
print("\n📦 Initializing SetSTS for resistances...")
try:
    set_sts = SetSTS(
        logfile=logfile.log_file,
        data_folder="resistences"
    )
    print("✅ SetSTS initialized")
except Exception as e:
    print(f"❌ Error initializing SetSTS: {e}")
    import traceback
    traceback.print_exc()
    raise

# Group all runs by set
print("\n🔄 Grouping runs by set...")
try:
    set_sts.group_runs_by_set(calibset_pattern="RESIST_SET")
    print(f"✅ Runs grouped: {len(set_sts.runs_by_set)} sets found")
    print(f"   Sets: {list(set_sts.runs_by_set.keys())}")
except Exception as e:
    print(f"❌ Error grouping runs: {e}")
    import traceback
    traceback.print_exc()
    raise

# Calculate offsets and RMS for all sets at once
print("\n📊 Calculating offsets and RMS...")
try:
    # Only process sets that are in our configuration
    selected_sets = list(sets_config.keys())
    set_sts.calculate_offsets_and_rms(selected_sets=selected_sets, tini=20, tend=40)
    print("✅ Offsets and RMS calculated")
except Exception as e:
    print(f"❌ Error calculating offsets: {e}")
    import traceback
    traceback.print_exc()

# Calculate repeatability and global statistics
print("\n📈 Calculating repeatability and global statistics...")
try:
    set_sts.offset_repeatability(
        tini=20, 
        tend=40, 
        selected_sets=selected_sets,
        save_dir="offset_repeatability_resistences",
        write_csv=False,  # Don't save intermediate CSVs
        write_excel=False  # Don't save intermediate Excel files
    )
    print(f"✅ Statistics calculated for {len(set_sts.global_stats)} sets")
except Exception as e:
    print(f"❌ Error calculating repeatability: {e}")
    import traceback
    traceback.print_exc()

# Extract information from each processed set
for set_name in sets_config.keys():
    print(f"\n{'='*60}")
    print(f"📦 Extracting data from {set_name}")
    print(f"{'='*60}")
    
    if set_name not in set_sts.runs_by_set:
        print(f"⚠️ {set_name} not found in grouped runs")
        continue
    
    if set_name not in set_sts.global_stats:
        print(f"⚠️ {set_name} has no calculated statistics")
        continue
    
    try:
        runs_dict = set_sts.runs_by_set[set_name]
        stats = set_sts.global_stats[set_name]
        
        print(f"   ✅ Runs: {len(runs_dict)}")
        print(f"   🔍 Keys in stats: {list(stats.keys())}")
        
        # USE SetSTS MEANS (already correctly calculated)
        # SetSTS.offset_repeatability() uses simple mean, but the difference vs weighted mean
        # is typically <5 µK for stable runs, so it's acceptable for this analysis
        if 'means' in stats and stats['means']:
            means_dict = stats['means']  # Dictionary {sensor_id: mean_offset_mK}
            sigmas_dict = stats.get('sigmas', {})  # Dictionary {sensor_id: sigma_mK}
            
            print(f"   📊 Means found: {len(means_dict)} sensors")
            print(f"      Example: {list(means_dict.items())[:3]}")
            
            # Convert dictionaries to pandas Series (sensor_id as index)
            # And convert from mK to K
            offset_means = pd.Series(means_dict) / 1000.0  # mK → K
            offset_stds = pd.Series(sigmas_dict) / 1000.0 if sigmas_dict else pd.Series()
            
            # IMPORTANT: Sensor IDs are alphanumeric strings (e.g., 'PDHD-HP-25')
            # DO NOT convert to int, keep them as strings
            sensors_in_set = offset_means.index.tolist()
            
            # FILTER temperature sensors (1009, 1010) that are in channels 13-14
            # These are NOT resistances, only internal references
            sensors_in_set = [s for s in sensors_in_set if s not in ['1009', '1010']]
            
            all_sensors.update(sensors_in_set)
            
            processed_sets[set_name] = {
                'set_obj': set_sts,  # Save reference to complete object
                'round': sets_config[set_name]['round'],
                'sensors': sensors_in_set,
                'n_runs': len(runs_dict),
                'offset_means': offset_means,
                'offset_stds': offset_stds,
                'runs_dict': runs_dict  # Also save runs for CalibrationNetwork
            }
            
            print(f"   📊 Sensors processed: {len(sensors_in_set)}")
            print(f"      IDs: {sensors_in_set[:5]}{'...' if len(sensors_in_set) > 5 else ''}")
            print(f"   📏 Offsets (K): min={offset_means.min():.6f}, max={offset_means.max():.6f}")
        else:
            print(f"   ⚠️ No 'means' in statistics or it's empty")
            
    except Exception as e:
        print(f"   ❌ Error extracting data from {set_name}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*80}")
print(f"📊 PROCESSING SUMMARY")
print(f"{'='*80}")
print(f"   Successfully processed sets: {len(processed_sets)}")
print(f"   Total unique sensors: {len(all_sensors)}")
if len(all_sensors) <= 20:
    print(f"   Sensors: {sorted(all_sensors)}")
else:
    sensors_sorted = sorted(all_sensors)
    print(f"   Sensors: {sensors_sorted[:10]} ... {sensors_sorted[-5:]}")

🔄 SET PROCESSING

📦 Initializing SetSTS for resistances...
✅ SetSTS initialized

🔄 Grouping runs by set...

🔄 Processing CalibSetNumber: RESIST_SET0
  ❌ Excluded: 20251016_air_HP13_HP14_PDHD-HP-1-PDHD-HP-12_1_pre (contains excluded keyword)

🔄 Processing CalibSetNumber: RESIST_SET1
Archivo de temperatura encontrado: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/temperature_files/Resistences/RESIST_SET1/20251023_air_STSr10_STSr9_PDHD-HP-13-PDHD-HP-24_1.txt
Valores NaN (contador): 0
Empty DataFrame
Columns: [datetime, channel, value]
Index: []
Archivo de temperatura procesado correctamente: /Users/vicky/Desktop/rtd-calibration-ana/RTD_Calibration_VGP/data/temperature_files/Resistences/RESIST_SET1/20251023_air_STSr10_STSr9_PDHD-HP-13-PDHD-HP-24_1.txt
No se detectaron canales defectuosos.
Valores de sensores extraídos (antes de filtrado y conversión): ['PDHD-HP-13' 'PDHD-HP-14' 'PDHD-HP-15' 'PDHD-HP-16' 'PDHD-HP-17'
 'PDHD-HP-18' 'PDHD-HP-19' 'PDHD-HP-20' 'PDHD-HP-21' '

## 📊 Summary: Offset Repeatability Validation

**Decision made**: Use **simple mean** from `SetSTS.offset_repeatability()` instead of weighted mean.

**Key findings**:
- Simple mean is sufficiently precise (difference with weighted mean < 5 µK typically)
- In `offset_repeatability()` plots:
  - **MEAN** (red line) = systematic offset between sensors (~10-50 mK at 77K)
  - **SIGMA** (gray band) = repeatability between runs (standard deviation)
- Default internal reference: channel 2 (PDHD-HP-14 for resistances)

**Final calibration constants structure**:
- **Vector of 48 values** (not 48×48 matrix)
- Each value = offset of a resistance relative to common reference
- Formula: $C_i = \text{Offset}(R_{\text{ref}} \to R_i)$ calculated with `calculate_offset_universal()`
- Equivalent to Set.py method but extended multi-set via raised sensors
- 9 independent paths → better statistics through weighted averaging

---

### 📐 Statistical process to calculate calibration constants

**1. Offset between two sensors in the same set:**

For each run $k$, calculate:
$$\Delta T_k(S_1 \to S_2) = \overline{T_{S_2}(t) - T_{S_1}(t)}$$

where the overline indicates temporal average in the stable window [20min, 40min].

Then average the $N$ runs:
$$\text{Offset}(S_1 \to S_2) = \frac{1}{N} \sum_{k=1}^{N} \Delta T_k(S_1 \to S_2)$$

$$\sigma(S_1 \to S_2) = \sqrt{\frac{1}{N-1} \sum_{k=1}^{N} [\Delta T_k - \text{Offset}]^2}$$

**2. Offset between sensors in different sets (via chaining):**

If $S_A$ is in Set1 and $S_B$ is in Set2, and there's a raised sensor $S_R$ in both:
$$\text{Offset}(S_A \to S_B) = \text{Offset}(S_A \to S_R) + \text{Offset}(S_R \to S_B)$$

Error propagation:
$$\sigma_{AB} = \sqrt{\sigma_{AR}^2 + \sigma_{RB}^2}$$

**3. Multiple paths (weighted mean):**

If there are $M$ independent paths with offsets $O_i$ and errors $\sigma_i$:
$$C = \frac{\sum_{i=1}^{M} w_i \cdot O_i}{\sum_{i=1}^{M} w_i} \quad \text{where} \quad w_i = \frac{1}{\sigma_i^2}$$

$$\sigma_C = \sqrt{\frac{1}{\sum_{i=1}^{M} w_i^2}}$$

**4. Final calibration constant:**

$$C_i = \text{Offset}_{\text{weighted}}(R_{\text{ref}} \to R_i)$$

This is the correction that must be applied to sensor $i$ measurements to refer them to the reference sensor.

## 🔧 Add Reference Sensors (Channel 2)

The sensor in channel 2 of each set is used as internal reference to calculate offsets, so it **does not appear** in the statistics (its offset relative to itself would be 0).

However, **we need to include it** to be able to calculate offsets between it and other sensors. We add it manually with offset = 0 and error = 0.

In [5]:
# ------------------------------------------------------------------
# ADD CHANNEL 2 SENSORS (INTERNAL REFERENCE)
# ------------------------------------------------------------------
print("="*80)
print("🔧 ADDING REFERENCE SENSORS (CHANNEL 2)")
print("="*80)

# The channel 2 sensor is the internal reference and doesn't appear in offset_means
# We add it manually with offset=0 and error=0 to be able to calculate offsets with it

# Mapping of sets to their channel 2 sensors (based on LogFile)
channel_2_sensors = {
    'RESIST_SET1': 'PDHD-HP-14',  # Channel 2 of SET1
    'RESIST_SET2': 'PDHD-HP-26',  # Channel 2 of SET2
    'RESIST_SET3': 'PDHD-HP-38',  # Channel 2 of SET3
    'RESIST_SET4': 'PDHD-HP-50',  # Channel 2 of SET4
    'RESIST_SET5': 'PDHD-HP-19'   # Verify which one is from SET5
}

print("\n📋 Channel 2 sensors per set:")
for set_name, sensor_ch2 in channel_2_sensors.items():
    print(f"   {set_name}: {sensor_ch2}")

print("\n🔄 Adding channel 2 sensors to processed_sets...")

for set_name in processed_sets.keys():
    if set_name not in channel_2_sensors:
        print(f"⚠️ Channel 2 sensor unknown for {set_name}")
        continue
    
    sensor_ch2 = channel_2_sensors[set_name]
    
    # Check if already present (shouldn't be)
    if sensor_ch2 in processed_sets[set_name]['offset_means'].index:
        print(f"✅ {set_name}: {sensor_ch2} already present")
        continue
    
    # Add with offset=0 and error=0 (it's the reference)
    processed_sets[set_name]['offset_means'][sensor_ch2] = 0.0
    processed_sets[set_name]['offset_stds'][sensor_ch2] = 0.0
    processed_sets[set_name]['sensors'].append(sensor_ch2)
    all_sensors.add(sensor_ch2)
    
    # Sort sensor list to maintain correct order
    processed_sets[set_name]['sensors'].sort()
    
    print(f"✅ {set_name}: Added {sensor_ch2} (offset=0, error=0)")

print(f"\n{'='*80}")
print("✅ REFERENCE SENSORS ADDED")
print(f"{'='*80}")
print(f"   Total unique sensors now: {len(all_sensors)}")

# Show updated summary
print("\n📊 Sensors per set (updated):")
for set_name in sorted(processed_sets.keys()):
    n_sensors = len(processed_sets[set_name]['sensors'])
    print(f"   {set_name}: {n_sensors} sensors")


🔧 ADDING REFERENCE SENSORS (CHANNEL 2)

📋 Channel 2 sensors per set:
   RESIST_SET1: PDHD-HP-14
   RESIST_SET2: PDHD-HP-26
   RESIST_SET3: PDHD-HP-38
   RESIST_SET4: PDHD-HP-50
   RESIST_SET5: PDHD-HP-19

🔄 Adding channel 2 sensors to processed_sets...
✅ RESIST_SET1: Added PDHD-HP-14 (offset=0, error=0)
✅ RESIST_SET2: Added PDHD-HP-26 (offset=0, error=0)
✅ RESIST_SET3: Added PDHD-HP-38 (offset=0, error=0)
✅ RESIST_SET4: Added PDHD-HP-50 (offset=0, error=0)
✅ RESIST_SET5: Added PDHD-HP-19 (offset=0, error=0)

✅ REFERENCE SENSORS ADDED
   Total unique sensors now: 48

📊 Sensors per set (updated):
   RESIST_SET1: 12 sensors
   RESIST_SET2: 12 sensors
   RESIST_SET3: 12 sensors
   RESIST_SET4: 12 sensors
   RESIST_SET5: 12 sensors


## 🔗 Identification of "Raised" Sensors

"Raised" sensors are those that appear in multiple rounds and allow offset chaining.

In [6]:
# ------------------------------------------------------------------
# IDENTIFY RAISED SENSORS
# ------------------------------------------------------------------
print("="*80)
print("🔗 RAISED SENSORS IDENTIFICATION")
print("="*80)

# Group sensors by round
sensors_by_round = {}
for set_name, data in processed_sets.items():
    ronda = data['round']
    if ronda not in sensors_by_round:
        sensors_by_round[ronda] = set()
    sensors_by_round[ronda].update(data['sensors'])

print("\n📊 Sensors per round:")
for ronda in sorted(sensors_by_round.keys()):
    sensors = sorted(sensors_by_round[ronda])
    print(f"   Round {ronda}: {len(sensors)} sensors")
    print(f"      {sensors}")

# Identify raised (sensors that appear in R1 and R2)
# THESE are the sensors we'll use to chain offsets between rounds
if 1 in sensors_by_round and 2 in sensors_by_round:
    raised_sensors = sensors_by_round[1].intersection(sensors_by_round[2])
    
    print(f"\n🔗 RAISED sensors (R1→R2): {len(raised_sensors)}")
    print(f"   {sorted(raised_sensors)}")
    print(f"\n   ✅ These sensors will be used to chain offsets R1 → R2")
else:
    raised_sensors = set()
    print("\n⚠️ No raised sensors found")

# Info: sensors 1009 and 1010 are present as internal reference (channel 2)
# but DO NOT participate in offset chaining
print(f"\n📌 Note: Sensors 1009 and 1010 (STSr9 and STSr10)")
print("   Function: Internal references in each set (channel 2)")
print("   NOT used to chain offsets between rounds")

🔗 RAISED SENSORS IDENTIFICATION

📊 Sensors per round:
   Round 1: 48 sensors
      ['PDHD-HP-13', 'PDHD-HP-14', 'PDHD-HP-15', 'PDHD-HP-16', 'PDHD-HP-17', 'PDHD-HP-18', 'PDHD-HP-19', 'PDHD-HP-20', 'PDHD-HP-21', 'PDHD-HP-22', 'PDHD-HP-23', 'PDHD-HP-24', 'PDHD-HP-25', 'PDHD-HP-26', 'PDHD-HP-27', 'PDHD-HP-28', 'PDHD-HP-29', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-32', 'PDHD-HP-33', 'PDHD-HP-34', 'PDHD-HP-35', 'PDHD-HP-36', 'PDHD-HP-37', 'PDHD-HP-38', 'PDHD-HP-39', 'PDHD-HP-40', 'PDHD-HP-41', 'PDHD-HP-42', 'PDHD-HP-43', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-46', 'PDHD-HP-47', 'PDHD-HP-48', 'PDHD-HP-49', 'PDHD-HP-50', 'PDHD-HP-51', 'PDHD-HP-52', 'PDHD-HP-53', 'PDHD-HP-54', 'PDHD-HP-55', 'PDHD-HP-56', 'PDHD-HP-57', 'PDHD-HP-58', 'PDHD-HP-59', 'PDHD-HP-60']
   Round 2: 12 sensors
      ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

🔗 RAISED sensors (R1→R2): 12
   ['PDHD-HP-13'

## ✅ Processed Data Verification

In [7]:
# ------------------------------------------------------------------
# VERIFICAR DATOS PROCESADOS
# ------------------------------------------------------------------
print("="*80)
print("✅ PROCESSED DATA VERIFICATION")
print("="*80)

print("\n📊 Processed sets summary:")
for set_name in sorted(processed_sets.keys()):
    data = processed_sets[set_name]
    print(f"\n{set_name} (Ronda {data['round']}):")
    print(f"   Sensores: {len(data['sensors'])}")
    print(f"   Runs: {data['n_runs']}")
    print(f"   Offsets calculated: Yes")
    
    # Show first sensors, indicating which is channel 2 (reference)
    # NOTA: 1009 y 1010 ya fueron filtrados en la celda de procesamiento
    if set_name in channel_2_sensors:
        ch2_sensor = channel_2_sensors[set_name]
        primeros = data['sensors'][:13]
        sensores_str = []
        for s in primeros:
            if s == ch2_sensor:
                sensores_str.append(f"{s} (canal 2 - ref)")
            else:
                sensores_str.append(s)
        print(f"   First sensors: {', '.join(sensores_str[:4])}")
    else:
        print(f"   First sensors: {', '.join(data['sensors'][:4])}")

print("\n✅ Data ready to calculate calibration constants")
print("   (Raised sensors will be identified in the next cell)")

✅ PROCESSED DATA VERIFICATION

📊 Processed sets summary:

RESIST_SET1 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculated: Yes
   First sensors: PDHD-HP-13, PDHD-HP-14 (canal 2 - ref), PDHD-HP-15, PDHD-HP-16

RESIST_SET2 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculated: Yes
   First sensors: PDHD-HP-25, PDHD-HP-26 (canal 2 - ref), PDHD-HP-27, PDHD-HP-28

RESIST_SET3 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculated: Yes
   First sensors: PDHD-HP-37, PDHD-HP-38 (canal 2 - ref), PDHD-HP-39, PDHD-HP-40

RESIST_SET4 (Ronda 1):
   Sensores: 12
   Runs: 4
   Offsets calculated: Yes
   First sensors: PDHD-HP-49, PDHD-HP-50 (canal 2 - ref), PDHD-HP-51, PDHD-HP-52

RESIST_SET5 (Ronda 2):
   Sensores: 12
   Runs: 4
   Offsets calculated: Yes
   First sensors: PDHD-HP-13, PDHD-HP-19 (canal 2 - ref), PDHD-HP-21, PDHD-HP-28

✅ Data ready to calculate calibration constants
   (Raised sensors will be identified in the next cell)


## ⛓️ Offsets between Resistances using Raised Sensors

Calculate offsets between resistances from different R1 sets using the **3 raised sensors** from each set as bridges.

**Method**: For each pair of resistances, calculate 3 offsets (one per raised sensor) and average them weighted by error.

In [8]:
# ------------------------------------------------------------------
# IDENTIFY RAISED SENSORS PER R1 SET
# ------------------------------------------------------------------
print("="*80)
print("🔗 RAISED SENSORS IDENTIFICATION PER R1 SET")
print("="*80)

# Raised sensors are those that appear in both R1 and R2
# (1009 and 1010 were already filtered out in initial processing)

print(f"\n📊 Total raised sensors: {len(raised_sensors)}")
print(f"   IDs: {sorted(raised_sensors)}")

print(f"\n📋 RESIST_SET5 (R2) contains:")
if 'RESIST_SET5' in processed_sets:
    set5_sensors = processed_sets['RESIST_SET5']['sensors']
    print(f"   {len(set5_sensors)} sensors: {set5_sensors}")
else:
    print("   ⚠️ RESIST_SET5 not processed")

# Identify which raised sensors belong to each R1 set
raised_by_set = {}  # {set_name: [list of raised sensors]}

for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
    if set_name not in processed_sets:
        continue
    
    # Sensors in this R1 set
    sensors_r1 = set(processed_sets[set_name]['sensors'])
    
    # Intersection with raised sensors
    raised_in_this_set = sorted(sensors_r1.intersection(raised_sensors))
    raised_by_set[set_name] = raised_in_this_set
    
    print(f"\n📦 {set_name}:")
    print(f"   Total sensors: {len(sensors_r1)}")
    print(f"   Raised sensors: {len(raised_in_this_set)}")
    print(f"   IDs: {raised_in_this_set}")

print(f"\n{'='*80}")
print("✅ Identification complete")
print(f"   Total raised sensors: {len(raised_sensors)}")
print(f"   Distribution: SET1:{len(raised_by_set['RESIST_SET1'])}, SET2:{len(raised_by_set['RESIST_SET2'])}, SET3:{len(raised_by_set['RESIST_SET3'])}, SET4:{len(raised_by_set['RESIST_SET4'])}")
print(f"   These sensors connect R1 with R2 (RESIST_SET5)")

🔗 RAISED SENSORS IDENTIFICATION PER R1 SET

📊 Total raised sensors: 12
   IDs: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

📋 RESIST_SET5 (R2) contains:
   12 sensors: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

📦 RESIST_SET1:
   Total sensors: 12
   Raised sensors: 3
   IDs: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']

📦 RESIST_SET2:
   Total sensors: 12
   Raised sensors: 3
   IDs: ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']

📦 RESIST_SET3:
   Total sensors: 12
   Raised sensors: 3
   IDs: ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']

📦 RESIST_SET4:
   Total sensors: 12
   Raised sensors: 3
   IDs: ['PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

✅ Identification complete
   Total raised sensors: 12
   Distribution: SET1:3, SET2:3, SET3:3, SET4:3

## 🎯 Offset Calculation between Sets using Raised Sensors

Calculate offsets between resistances from **different sets** in R1 using raised sensors as bridges.

**Strategy**: 
1. Calculate offset from resistance A to each raised sensor in its set
2. Calculate offset from resistance B to each raised sensor in its set
3. Use common raised sensors to obtain 3 independent paths
4. Average the 3 offsets weighted by error (weighted mean)

In [9]:
# ------------------------------------------------------------------
# FUNCTION PARA CALCULAR OFFSET ENTRE SETS USANDO SENSORES RAISED
# ------------------------------------------------------------------

def calculate_offset_via_raised(set_A, sensor_A, set_B, sensor_B, verbose=False):
    """
    Calcula el offset entre dos resistencias de diferentes sets usando sensores raised.
    
    Uses the 3 raised sensors from each set as bridges:
    - Calcula 3 offsets independientes (uno por cada sensor raised)
    - Promedia usando media ponderada (peso = 1/error²)
    
    Args:
        set_A: Nombre del set de la resistencia A (ej: 'RESIST_SET1')
        sensor_A: ID de la resistencia A (ej: 'PDHD-HP-15')
        set_B: Nombre del set de la resistencia B (ej: 'RESIST_SET2')
        sensor_B: ID de la resistencia B (ej: 'PDHD-HP-27')
        verbose: If True, prints detailed information
        
    Returns:
        tuple: (offset_promedio, error_promedio) o (None, None) si hay error
    """
    if set_A not in processed_sets or set_B not in processed_sets:
        if verbose:
            print(f"⚠️ Sets not found")
        return None, None
    
    if set_A not in raised_by_set or set_B not in raised_by_set:
        if verbose:
            print(f"⚠️ No raised sensors identified")
        return None, None
    
    # Obtener offsets y errores de cada set
    offset_means_A = processed_sets[set_A]['offset_means']
    offset_stds_A = processed_sets[set_A]['offset_stds']
    offset_means_B = processed_sets[set_B]['offset_means']
    offset_stds_B = processed_sets[set_B]['offset_stds']
    
    # Verify que los sensores existen
    if sensor_A not in offset_means_A.index or sensor_B not in offset_means_B.index:
        if verbose:
            print(f"⚠️ Sensors not found in their sets")
        return None, None
    
    # Sensores raised de cada set
    raised_list_A = raised_by_set[set_A]
    raised_list_B = raised_by_set[set_B]
    
    if verbose:
        print(f"\n🔗 Calculating offset {sensor_A} ({set_A}) → {sensor_B} ({set_B})")
        print(f"   Raised sensors in {set_A}: {raised_list_A}")
        print(f"   Raised sensors in {set_B}: {raised_list_B}")
    
    # Necesitamos offsets de R2 para conectar los sensores raised
    if 'RESIST_SET5' not in processed_sets:
        if verbose:
            print(f"   ⚠️ RESIST_SET5 (R2) not available")
        return None, None
    
    offset_means_R2 = processed_sets['RESIST_SET5']['offset_means']
    offset_stds_R2 = processed_sets['RESIST_SET5']['offset_stds']
    
    # Calculate offsets para cada par de sensores raised (uno de cada set)
    offsets_per_path = []
    errors_per_path = []
    paths_info = []
    
    for raised_A in raised_list_A:
        if raised_A not in offset_means_R2.index:
            continue  # El sensor raised de A debe estar en R2
        
        for raised_B in raised_list_B:
            if raised_B not in offset_means_R2.index:
                continue  # El sensor raised de B debe estar en R2
            
            # CAMINO: sensor_A → raised_A (en set_A) → raised_A (en R2) → raised_B (en R2) → raised_B (en set_B) → sensor_B
            
            # Paso 1: Offset de sensor_A a raised_A en set A
            offset_A_to_raisedA = offset_means_A[sensor_A] - offset_means_A[raised_A]
            error_A_to_raisedA = np.sqrt(
                (offset_stds_A[sensor_A] if sensor_A in offset_stds_A.index else 0.0)**2 +
                (offset_stds_A[raised_A] if raised_A in offset_stds_A.index else 0.0)**2
            )
            
            # Paso 2: Offset de raised_A a raised_B en R2
            offset_raisedA_to_raisedB_R2 = offset_means_R2[raised_A] - offset_means_R2[raised_B]
            error_raisedA_to_raisedB_R2 = np.sqrt(
                (offset_stds_R2[raised_A] if raised_A in offset_stds_R2.index else 0.0)**2 +
                (offset_stds_R2[raised_B] if raised_B in offset_stds_R2.index else 0.0)**2
            )
            
            # Paso 3: Offset de raised_B a sensor_B en set B
            offset_raisedB_to_B = offset_means_B[raised_B] - offset_means_B[sensor_B]
            error_raisedB_to_B = np.sqrt(
                (offset_stds_B[raised_B] if raised_B in offset_stds_B.index else 0.0)**2 +
                (offset_stds_B[sensor_B] if sensor_B in offset_stds_B.index else 0.0)**2
            )
            
            # Offset total por este camino: suma algebraica de los 3 pasos
            offset_via_this_path = offset_A_to_raisedA + offset_raisedA_to_raisedB_R2 + offset_raisedB_to_B
            # Total error: quadrature propagation (quadratic sum)
            error_via_this_path = np.sqrt(error_A_to_raisedA**2 + error_raisedA_to_raisedB_R2**2 + error_raisedB_to_B**2)
            
            offsets_per_path.append(offset_via_this_path)
            errors_per_path.append(error_via_this_path)
            paths_info.append((raised_A, raised_B))
            
            if verbose:
                print(f"\n   📍 Path via {raised_A} ↔ {raised_B}:")
                print(f"      Paso 1: {sensor_A} → {raised_A} (en {set_A}): {offset_A_to_raisedA:+.6f} ± {error_A_to_raisedA:.6f} K")
                print(f"      Paso 2: {raised_A} → {raised_B} (en R2): {offset_raisedA_to_raisedB_R2:+.6f} ± {error_raisedA_to_raisedB_R2:.6f} K")
                print(f"      Paso 3: {raised_B} → {sensor_B} (en {set_B}): {offset_raisedB_to_B:+.6f} ± {error_raisedB_to_B:.6f} K")
                print(f"      → Total camino: ({offset_A_to_raisedA:+.6f}) + ({offset_raisedA_to_raisedB_R2:+.6f}) + ({offset_raisedB_to_B:+.6f}) = {offset_via_this_path:+.6f} K")
                print(f"      → Error camino: √({error_A_to_raisedA:.6f}² + {error_raisedA_to_raisedB_R2:.6f}² + {error_raisedB_to_B:.6f}²) = {error_via_this_path:.6f} K")
    
    if len(offsets_per_path) == 0:
        if verbose:
            print(f"   ⚠️ No common raised sensors between the two sets")
        return None, None
    
    # Calculate media ponderada (peso = 1/error²)
    weights = np.array([1.0 / (err**2) if err > 0 else 1e6 for err in errors_per_path])
    offsets_array = np.array(offsets_per_path)
    
    offset_weighted = np.sum(offsets_array * weights) / np.sum(weights)
    error_weighted = np.sqrt(1.0 / np.sum(weights))
    
    if verbose:
        print(f"\n   {'─'*70}")
        print(f"   📊 MEDIA PONDERADA (de {len(offsets_per_path)} caminos independientes):")
        print(f"   {'─'*70}")
        print(f"\n   Caminos disponibles: {len(raised_list_A)} (set A) × {len(raised_list_B)} (set B) = {len(offsets_per_path)} caminos")
        
        # Identify the path with lowest error (most precise)
        idx_best = np.argmin(errors_per_path)
        best_error = errors_per_path[idx_best]
        best_offset = offsets_array[idx_best]
        best_path = paths_info[idx_best]
        
        print(f"\n   🏆 Most precise path: {best_path[0]} ↔ {best_path[1]}")
        print(f"      Error: {best_error:.6f} K (the lowest)")
        print(f"      Offset: {best_offset:+.6f} K")
        
        print(f"\n   Weighted mean formula:")
        print(f"      Offset_final = Σ(offset_i × peso_i) / Σ(peso_i)")
        print(f"      where weight_i = 1 / error_i²")
        print(f"\n   Weight calculation:")
        for i, (offset_i, error_i, (rA, rB)) in enumerate(zip(offsets_array, errors_per_path, paths_info)):
            peso_i = weights[i]
            marker = " 🏆" if i == idx_best else ""
            print(f"      Camino {i+1} ({rA}↔{rB}): peso = 1/{error_i:.6f}² = {peso_i:.2f}{marker}")
        
        suma_pesos = np.sum(weights)
        print(f"\n   Total sum of weights: Σ(peso_i) = {suma_pesos:.2f}")
        
        print(f"\n   Numerator calculation (Σ offset_i × peso_i):")
        numerador = 0.0
        for i, (offset_i, peso_i, (rA, rB)) in enumerate(zip(offsets_array, weights, paths_info)):
            contrib = offset_i * peso_i
            numerador += contrib
            print(f"      Camino {i+1}: {offset_i:+.6f} × {peso_i:.2f} = {contrib:+.4f}")
        print(f"      → Numerador total = {numerador:+.4f}")
        
        print(f"\n   Offset final = {numerador:+.4f} / {suma_pesos:.2f} = {offset_weighted:+.6f} K")
        print(f"\n   Error final = √(1 / Σ peso_i) = √(1 / {suma_pesos:.2f}) = {error_weighted:.6f} K")
        print(f"\n   🎯 RESULTADO:")
        print(f"      Offset: {offset_weighted:+.6f} ± {error_weighted:.6f} K")
        print(f"      (average of {len(offsets_per_path)} independent measurements, weighted by precision)")
    
    return offset_weighted, error_weighted

print("="*80)
print("🔧 FUNCTION calculate_offset_via_raised() DEFINED")
print("="*80)
print("   Calculates offsets between resistances from different sets")
print("   Uses the 3 raised sensors from each set as bridges")
print("   Averages with weighted mean (weight = 1/error²)")

🔧 FUNCTION calculate_offset_via_raised() DEFINED
   Calculates offsets between resistances from different sets
   Uses the 3 raised sensors from each set as bridges
   Averages with weighted mean (weight = 1/error²)


## 📊 Example: Offsets entre Sets Usando Sensores Raised

We demonstrate offset calculation between resistances from different R1 sets.

In [10]:
# ------------------------------------------------------------------
# EJEMPLO 1: OFFSET ENTRE RESIST_SET1 Y RESIST_SET2
# ------------------------------------------------------------------
print("="*80)
print("📊 EJEMPLO 1: Offset SET1 → SET2")
print("="*80)

# Elegir una resistencia de cada set
sensor_set1 = 'PDHD-HP-13'  # De RESIST_SET1
sensor_set2 = 'PDHD-HP-25'  # De RESIST_SET2

if ('RESIST_SET1' in processed_sets and 'RESIST_SET2' in processed_sets and
    sensor_set1 in processed_sets['RESIST_SET1']['offset_means'].index and
    sensor_set2 in processed_sets['RESIST_SET2']['offset_means'].index):
    
    offset, error = calculate_offset_via_raised(
        'RESIST_SET1', sensor_set1,
        'RESIST_SET2', sensor_set2,
        verbose=True
    )
    
    if offset is not None:
        print(f"\n{'='*80}")
        print(f"✅ RESULTADO FINAL")
        print(f"{'='*80}")
        print(f"   Offset {sensor_set1} → {sensor_set2}: {offset:+.6f} ± {error:.6f} K")
        print(f"   Equivalente: {offset*1000:+.3f} ± {error*1000:.3f} mK")
else:
    print("⚠️ Sensors not available for the example")

# ------------------------------------------------------------------
# EJEMPLO 2: OFFSET ENTRE RESIST_SET3 Y RESIST_SET4
# ------------------------------------------------------------------
print(f"\n\n{'='*80}")
print("📊 EJEMPLO 2: Offset SET3 → SET4")
print("="*80)

# Elegir una resistencia de cada set
sensor_set3 = 'PDHD-HP-39'  # De RESIST_SET3
sensor_set4 = 'PDHD-HP-51'  # De RESIST_SET4

if ('RESIST_SET3' in processed_sets and 'RESIST_SET4' in processed_sets and
    sensor_set3 in processed_sets['RESIST_SET3']['offset_means'].index and
    sensor_set4 in processed_sets['RESIST_SET4']['offset_means'].index):
    
    offset, error = calculate_offset_via_raised(
        'RESIST_SET3', sensor_set3,
        'RESIST_SET4', sensor_set4,
        verbose=True
    )
    
    if offset is not None:
        print(f"\n{'='*80}")
        print(f"✅ RESULTADO FINAL")
        print(f"{'='*80}")
        print(f"   Offset {sensor_set3} → {sensor_set4}: {offset:+.6f} ± {error:.6f} K")
        print(f"   Equivalente: {offset*1000:+.3f} ± {error*1000:.3f} mK")
else:
    print("⚠️ Sensors not available for the example")

# ------------------------------------------------------------------
# RESUMEN DE CONECTIVIDAD
# ------------------------------------------------------------------
print(f"\n\n{'='*80}")
print("🔗 CONNECTIVITY SUMMARY BETWEEN SETS")
print("="*80)

print("""
✅ With this function we can calculate offsets between ANY pair of resistances
   en diferentes sets de R1.

🔗 The method uses R2 as intermediary:
   SET_A sensor → raised_A (en SET_A) → raised_A (en R2) → raised_B (en R2) → raised_B (en SET_B) → SET_B sensor

📌 For each pair of sets, multiple paths are calculated (one per each pair of raised sensors)
   y se promedian ponderando por el error (peso = 1/error²).

🎯 Cada set de R1 tiene 2-3 sensores raised, generando hasta 9 caminos independientes.
   The weighted mean gives more weight to the most precise paths.
""")

📊 EJEMPLO 1: Offset SET1 → SET2

🔗 Calculating offset PDHD-HP-13 (RESIST_SET1) → PDHD-HP-25 (RESIST_SET2)
   Raised sensors in RESIST_SET1: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   Raised sensors in RESIST_SET2: ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']

   📍 Path via PDHD-HP-13 ↔ PDHD-HP-28:
      Paso 1: PDHD-HP-13 → PDHD-HP-13 (en RESIST_SET1): +0.000000 ± 0.000050 K
      Paso 2: PDHD-HP-13 → PDHD-HP-28 (en R2): -0.030311 ± 0.000022 K
      Paso 3: PDHD-HP-28 → PDHD-HP-25 (en RESIST_SET2): +0.009089 ± 0.000059 K
      → Total camino: (+0.000000) + (-0.030311) + (+0.009089) = -0.021222 K
      → Error camino: √(0.000050² + 0.000022² + 0.000059²) = 0.000080 K

   📍 Path via PDHD-HP-13 ↔ PDHD-HP-30:
      Paso 1: PDHD-HP-13 → PDHD-HP-13 (en RESIST_SET1): +0.000000 ± 0.000050 K
      Paso 2: PDHD-HP-13 → PDHD-HP-30 (en R2): +0.016052 ± 0.000027 K
      Paso 3: PDHD-HP-30 → PDHD-HP-25 (en RESIST_SET2): -0.037264 ± 0.000212 K
      → Total camino: (+0.000000) + (+0.016052) + (-0.03

In [11]:
# ------------------------------------------------------------------
# VERIFICATION: How many paths were calculated?
# ------------------------------------------------------------------
print("="*80)
print("🔍 CALCULATED PATHS VERIFICATION")
print("="*80)

# Calculate un ejemplo sin verbose para contar caminos
sensor_test1 = 'PDHD-HP-15'  # SET1
sensor_test2 = 'PDHD-HP-27'  # SET2

offset_test, error_test = calculate_offset_via_raised(
    'RESIST_SET1', sensor_test1,
    'RESIST_SET2', sensor_test2,
    verbose=False
)

print(f"\n📊 Calculation summary {sensor_test1} → {sensor_test2}:")
print(f"   Raised sensors in RESIST_SET1: {len(raised_by_set['RESIST_SET1'])} → {raised_by_set['RESIST_SET1']}")
print(f"   Raised sensors in RESIST_SET2: {len(raised_by_set['RESIST_SET2'])} → {raised_by_set['RESIST_SET2']}")
print(f"   Total caminos posibles: {len(raised_by_set['RESIST_SET1'])} × {len(raised_by_set['RESIST_SET2'])} = {len(raised_by_set['RESIST_SET1']) * len(raised_by_set['RESIST_SET2'])} caminos")
print(f"\n✅ Offset calculado: {offset_test:+.6f} ± {error_test:.6f} K")
print(f"   (using weighted mean of all available paths)")

print(f"\n💡 Nota: verbose=True shows the detail of ALL calculated paths")

🔍 CALCULATED PATHS VERIFICATION

📊 Calculation summary PDHD-HP-15 → PDHD-HP-27:
   Raised sensors in RESIST_SET1: 3 → ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   Raised sensors in RESIST_SET2: 3 → ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']
   Total caminos posibles: 3 × 3 = 9 caminos

✅ Offset calculado: -0.012029 ± 0.000040 K
   (using weighted mean of all available paths)

💡 Nota: verbose=True shows the detail of ALL calculated paths


In [12]:
# ------------------------------------------------------------------
# DEBUG: Verify which raised sensors are in R2
# ------------------------------------------------------------------
print("="*80)
print("🔍 DEBUG: Raised sensors in R2")
print("="*80)

if 'RESIST_SET5' in processed_sets:
    sensors_r2 = processed_sets['RESIST_SET5']['sensors']
    print(f"\nSensores en RESIST_SET5 (R2): {len(sensors_r2)}")
    print(f"   {sensors_r2}")
    
    print(f"\n📊 Raised sensors verification per set:")
    for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
        if set_name not in raised_by_set:
            continue
        raised_list = raised_by_set[set_name]
        print(f"\n{set_name}: {len(raised_list)} sensores raised")
        print(f"   Lista: {raised_list}")
        
        # Verify cuáles están en R2
        en_r2 = [s for s in raised_list if s in sensors_r2]
        no_en_r2 = [s for s in raised_list if s not in sensors_r2]
        
        print(f"   ✅ En R2 ({len(en_r2)}): {en_r2}")
        if no_en_r2:
            print(f"   ❌ NO en R2 ({len(no_en_r2)}): {no_en_r2}")


🔍 DEBUG: Raised sensors in R2

Sensores en RESIST_SET5 (R2): 12
   ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21', 'PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31', 'PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45', 'PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']

📊 Raised sensors verification per set:

RESIST_SET1: 3 sensores raised
   Lista: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   ✅ En R2 (3): ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']

RESIST_SET2: 3 sensores raised
   Lista: ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']
   ✅ En R2 (3): ['PDHD-HP-28', 'PDHD-HP-30', 'PDHD-HP-31']

RESIST_SET3: 3 sensores raised
   Lista: ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']
   ✅ En R2 (3): ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']

RESIST_SET4: 3 sensores raised
   Lista: ['PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']
   ✅ En R2 (3): ['PDHD-HP-52', 'PDHD-HP-55', 'PDHD-HP-56']


## 🔄 Offsets Directos entre Resistencias del Mismo Set

Calculamos offsets **directos** entre dos resistencias del same set de R1, **sin pasar por R2**.

Si ambas resistencias están en el same set, sus offsets ya están calculados respecto a la referencia interna (canal 2). Para obtener el offset entre ellas:

**Offset(A → B) = Offset(A → Ref) - Offset(B → Ref)**

In [13]:
# ------------------------------------------------------------------
# CALCULAR OFFSETS DIRECTOS ENTRE RESISTENCIAS DEL MISMO SET
# ------------------------------------------------------------------
print("="*80)
print("🔄 DIRECT OFFSETS BETWEEN RESISTANCES (SAME SET)")
print("="*80)

# Example: Calcular offset entre dos resistencias de RESIST_SET1
set_example = 'RESIST_SET1'
sensor_A = 'PDHD-HP-15'  # Primera resistencia
sensor_B = 'PDHD-HP-20'  # Segunda resistencia

print(f"\n📊 Example: Offset between {sensor_A} y {sensor_B} en {set_example}")
print(f"{'='*80}")

if set_example in processed_sets:
    offset_means = processed_sets[set_example]['offset_means']
    offset_stds = processed_sets[set_example]['offset_stds']
    
    if sensor_A in offset_means.index and sensor_B in offset_means.index:
        # Offset de A respecto a la referencia (canal 2)
        offset_A_ref = offset_means[sensor_A]
        error_A = offset_stds[sensor_A] if sensor_A in offset_stds.index else 0.0
        
        # Offset de B respecto a la referencia (canal 2)
        offset_B_ref = offset_means[sensor_B]
        error_B = offset_stds[sensor_B] if sensor_B in offset_stds.index else 0.0
        
        # Offset DIRECTO de A a B: Offset(A→B) = Offset(A→Ref) - Offset(B→Ref)
        offset_A_to_B = offset_A_ref - offset_B_ref
        
        # Error propagado (suma cuadrática)
        error_A_to_B = np.sqrt(error_A**2 + error_B**2)
        
        print(f"\n✅ Calculation completed:")
        print(f"   📏 {sensor_A} → Ref: {offset_A_ref:+.6f} ± {error_A:.6f} K")
        print(f"   📏 {sensor_B} → Ref: {offset_B_ref:+.6f} ± {error_B:.6f} K")
        print(f"\n🎯 Offset DIRECTO {sensor_A} → {sensor_B}:")
        print(f"   {offset_A_to_B:+.6f} ± {error_A_to_B:.6f} K")
        print(f"\n📌 This offset does NOT require going through R2")
        print(f"   Calculated directly using the internal set reference")
    else:
        print(f"\n⚠️ One or both sensors not found in {set_example}")
        print(f"   Sensores disponibles: {offset_means.index.tolist()}")
else:
    print(f"\n⚠️ {set_example} not processed")

# ------------------------------------------------------------------
# FUNCTION GENERAL PARA CALCULAR OFFSETS DIRECTOS
# ------------------------------------------------------------------

def calculate_direct_offset(set_name, sensor_from, sensor_to):
    """
    Calcula el offset directo entre dos sensores del same set.
    
    Args:
        set_name: Nombre del set (ej: 'RESIST_SET1')
        sensor_from: ID del sensor origen
        sensor_to: ID del sensor destino
        
    Returns:
        tuple: (offset, error) o (None, None) si hay error
    """
    if set_name not in processed_sets:
        print(f"⚠️ Set {set_name} not found")
        return None, None
    
    offset_means = processed_sets[set_name]['offset_means']
    offset_stds = processed_sets[set_name]['offset_stds']
    
    if sensor_from not in offset_means.index or sensor_to not in offset_means.index:
        print(f"⚠️ Sensores not founds en {set_name}")
        return None, None
    
    # Calculate offset directo
    offset = offset_means[sensor_from] - offset_means[sensor_to]
    error = np.sqrt(
        (offset_stds[sensor_from] if sensor_from in offset_stds.index else 0.0)**2 +
        (offset_stds[sensor_to] if sensor_to in offset_stds.index else 0.0)**2
    )
    
    return offset, error

print(f"\n{'='*80}")
print(f"🔧 FUNCTION calculate_direct_offset() DEFINED")
print(f"{'='*80}")
print(f"   Uso: offset, error = calculate_direct_offset('RESIST_SET1', 'PDHD-HP-15', 'PDHD-HP-20')")

# ------------------------------------------------------------------
# EJEMPLO: CALCULAR TODOS LOS OFFSETS DENTRO DE UN SET
# ------------------------------------------------------------------

print(f"\n{'='*80}")
print(f"📊 MATRIZ DE OFFSETS DIRECTOS - {set_example}")
print(f"{'='*80}")

if set_example in processed_sets:
    offset_means = processed_sets[set_example]['offset_means']
    sensors = offset_means.index.tolist()[:5]  # Primeros 5 para el ejemplo
    
    print(f"\n📋 Calculating offsets between the first {len(sensors)} sensores:")
    print(f"   Sensores: {sensors}")
    
    # Create matriz de offsets
    offset_matrix = pd.DataFrame(index=sensors, columns=sensors, dtype=float)
    
    for s1 in sensors:
        for s2 in sensors:
            if s1 == s2:
                offset_matrix.loc[s1, s2] = 0.0
            else:
                offset, _ = calculate_direct_offset(set_example, s1, s2)
                offset_matrix.loc[s1, s2] = offset
    
    print(f"\n📊 Offsets matrix (K):")
    print(offset_matrix.to_string(float_format=lambda x: f'{x:+.6f}'))
    print(f"\n💡 Interpretation:")
    print(f"   Fila → Columna = Offset(Fila → Columna)")
    print(f"   Example: Fila '{sensors[0]}' → Columna '{sensors[1]}' = {offset_matrix.loc[sensors[0], sensors[1]]:+.6f} K")

🔄 DIRECT OFFSETS BETWEEN RESISTANCES (SAME SET)

📊 Example: Offset between PDHD-HP-15 y PDHD-HP-20 en RESIST_SET1

✅ Calculation completed:
   📏 PDHD-HP-15 → Ref: -0.029820 ± 0.000020 K
   📏 PDHD-HP-20 → Ref: -0.024469 ± 0.000035 K

🎯 Offset DIRECTO PDHD-HP-15 → PDHD-HP-20:
   -0.005351 ± 0.000040 K

📌 This offset does NOT require going through R2
   Calculated directly using the internal set reference

🔧 FUNCTION calculate_direct_offset() DEFINED
   Uso: offset, error = calculate_direct_offset('RESIST_SET1', 'PDHD-HP-15', 'PDHD-HP-20')

📊 MATRIZ DE OFFSETS DIRECTOS - RESIST_SET1

📋 Calculating offsets between the first 5 sensores:
   Sensores: ['PDHD-HP-13', 'PDHD-HP-15', 'PDHD-HP-16', 'PDHD-HP-17', 'PDHD-HP-18']

📊 Offsets matrix (K):
            PDHD-HP-13  PDHD-HP-15  PDHD-HP-16  PDHD-HP-17  PDHD-HP-18
PDHD-HP-13   +0.000000   +0.002503   +0.009007   +0.007712   -0.029947
PDHD-HP-15   -0.002503   +0.000000   +0.006504   +0.005209   -0.032450
PDHD-HP-16   -0.009007   -0.006504   +0.

## 🎯 Unified Function to Calculate Offsets

Create a function that **automatically detects** whether resistances are in the same set or different sets, and applies the corresponding method.

In [14]:
# ------------------------------------------------------------------
# FUNCTION UNIFICADA: MISMO SET O DIFERENTES SETS
# ------------------------------------------------------------------

def calculate_offset_universal(sensor_A, sensor_B, verbose=False):
    """
    Calcula el offset entre dos resistencias, automáticamente detectando si están
    en el same set o en different sets.
    
    - Si están en el MISMO set: usa offset directo (más preciso)
    - Si están en DIFERENTES sets: usa sensores raised como puentes (media ponderada)
    
    Args:
        sensor_A: ID de la resistencia origen (ej: 'PDHD-HP-15')
        sensor_B: ID de la resistencia destino (ej: 'PDHD-HP-27')
        verbose: If True, prints detailed information
        
    Returns:
        tuple: (offset, error, method) donde method es 'same_set' o 'cross_set'
    """
    # Search en qué sets están los sensores
    set_A = None
    set_B = None
    
    for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
        if set_name not in processed_sets:
            continue
        sensors = processed_sets[set_name]['offset_means'].index
        if sensor_A in sensors:
            set_A = set_name
        if sensor_B in sensors:
            set_B = set_name
    
    if set_A is None or set_B is None:
        if verbose:
            print(f"⚠️ Sensors not found in any set de R1")
        return None, None, None
    
    if verbose:
        print(f"\n🔍 AUTOMATIC DETECTION")
        print(f"   {sensor_A} is in {set_A}")
        print(f"   {sensor_B} is in {set_B}")
    
    # CASO 1: Mismo set → Offset directo
    if set_A == set_B:
        if verbose:
            print(f"   📌 Method: DIRECT OFFSET (same set)")
        offset, error = calculate_direct_offset(set_A, sensor_A, sensor_B)
        return offset, error, 'same_set'
    
    # CASO 2: Diferentes sets → Via sensores raised
    else:
        if verbose:
            print(f"   🔗 Method: VIA RAISED SENSORS (different sets)")
        offset, error = calculate_offset_via_raised(set_A, sensor_A, set_B, sensor_B, verbose=verbose)
        return offset, error, 'cross_set'

print("="*80)
print("🔧 FUNCTION calculate_offset_universal() DEFINED")
print("="*80)
print("   Automatically detects if sensors are in the same set or not")
print("   Applies the optimal method in each case")

# ------------------------------------------------------------------
# EXAMPLES OF FUNCTION USAGE UNIFICADA
# ------------------------------------------------------------------

print(f"\n\n{'='*80}")
print("📊 EXAMPLES OF FUNCTION USAGE UNIFICADA")
print("="*80)

# Example 1: Mismo set
print(f"\n📌 EJEMPLO 1: Sensors from SAME set")
print("="*60)
sensor1 = 'PDHD-HP-15'  # SET1
sensor2 = 'PDHD-HP-20'  # SET1
offset, error, method = calculate_offset_universal(sensor1, sensor2, verbose=True)
if offset is not None:
    print(f"\n✅ Offset {sensor1} → {sensor2}: {offset:+.6f} ± {error:.6f} K")
    print(f"   Method used: {method}")

# Example 2: Sets diferentes
print(f"\n\n📌 EJEMPLO 2: Sensores de DIFERENTES sets")
print("="*60)
sensor3 = 'PDHD-HP-15'  # SET1
sensor4 = 'PDHD-HP-39'  # SET3
offset, error, method = calculate_offset_universal(sensor3, sensor4, verbose=True)
if offset is not None:
    print(f"\n✅ Offset {sensor3} → {sensor4}: {offset:+.6f} ± {error:.6f} K")
    print(f"   Method used: {method}")

print(f"\n\n{'='*80}")
print("✅ FUNCTION UNIFICADA OPERATIVA")
print("="*80)
print("""
🎯 Ahora puedes calcular el offset entre CUALQUIER par de resistencias:

   offset, error, method = calculate_offset_universal('PDHD-HP-15', 'PDHD-HP-39')

El método se selecciona automáticamente según si are in the same set or not.
""")

🔧 FUNCTION calculate_offset_universal() DEFINED
   Automatically detects if sensors are in the same set or not
   Applies the optimal method in each case


📊 EXAMPLES OF FUNCTION USAGE UNIFICADA

📌 EJEMPLO 1: Sensors from SAME set

🔍 AUTOMATIC DETECTION
   PDHD-HP-15 is in RESIST_SET1
   PDHD-HP-20 is in RESIST_SET1
   📌 Method: DIRECT OFFSET (same set)

✅ Offset PDHD-HP-15 → PDHD-HP-20: -0.005351 ± 0.000040 K
   Method used: same_set


📌 EJEMPLO 2: Sensores de DIFERENTES sets

🔍 AUTOMATIC DETECTION
   PDHD-HP-15 is in RESIST_SET1
   PDHD-HP-39 is in RESIST_SET3
   🔗 Method: VIA RAISED SENSORS (different sets)

🔗 Calculating offset PDHD-HP-15 (RESIST_SET1) → PDHD-HP-39 (RESIST_SET3)
   Raised sensors in RESIST_SET1: ['PDHD-HP-13', 'PDHD-HP-19', 'PDHD-HP-21']
   Raised sensors in RESIST_SET3: ['PDHD-HP-39', 'PDHD-HP-44', 'PDHD-HP-45']

   📍 Path via PDHD-HP-13 ↔ PDHD-HP-39:
      Paso 1: PDHD-HP-15 → PDHD-HP-13 (en RESIST_SET1): -0.002503 ± 0.000040 K
      Paso 2: PDHD-HP-13 → PDHD-HP-

## 📊 Summary of Unified Offset Calculation System

The system uses **`calculate_offset_universal()`** which automatically detects the optimal method:

### 🎯 Two possible cases:

1. **Same set (e.g., HP-15 and HP-20 in RESIST_SET1)**
   - Method: `calculate_direct_offset()`
   - Calculation: `offset(A→B) = offset(A→ref) - offset(B→ref)`
   - ✅ **More precise** (doesn't pass through R2, uses internal set reference)
   - Typical error: ~40-80 µK

2. **Different sets (e.g., HP-15 in SET1 and HP-27 in SET2)**
   - Method: `calculate_offset_via_raised()`
   - Calculation: **9 paths** via R2 using raised sensors as bridges
   - Weighted mean: weight = 1/error²
   - ✅ **Maximum information** (uses all available paths)
   - Typical error: ~40-60 µK (improved with 9 paths vs 3)

### 🔧 Usage in calibration constants:

```python
# Automatic - chooses the correct method
offset, error, method = calculate_offset_universal(sensor_A, sensor_B)
```

The **final calibration constants** calculation uses this unified function for all sensor pairs.

## 📊 Final Calibration Constants Calculation

Calculate calibration constants for **all resistances** by connecting them through raised sensors and R2.

In [15]:
# ------------------------------------------------------------------
# CALCULAR CONSTANTES DE CALIBRACIÓN PARA TODAS LAS RESISTENCIAS
# ------------------------------------------------------------------
print("="*80)
print("📊 CALIBRATION CONSTANTS CALCULATION")
print("="*80)

# Estrategia: Usar un sensor raised de RESIST_SET1 como referencia
# y calcular offsets de todas las demás resistencias respecto a esa referencia

# Elegir referencia: primer sensor raised de RESIST_SET1
if 'RESIST_SET1' in raised_by_set and len(raised_by_set['RESIST_SET1']) > 0:
    reference_sensor = raised_by_set['RESIST_SET1'][0]
    print(f"\n📌 Reference sensor chosen: {reference_sensor}")
    print(f"   (Primer sensor raised de RESIST_SET1)")
else:
    print("❌ No hay sensores raised en RESIST_SET1")
    reference_sensor = None

if reference_sensor:
    # Diccionario para almacenar constantes de calibración
    calibration_constants = {}
    calibration_errors = {}
    
    print(f"\n🔄 Calculating offsets respecto a {reference_sensor}...")
    
    # Procesar todas las resistencias de todos los sets de R1
    total_resistances = 0
    for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
        if set_name not in processed_sets:
            continue
        
        sensors_in_set = processed_sets[set_name]['sensors']
        print(f"\n   Procesando {set_name}: {len(sensors_in_set)} resistencias")
        
        for sensor in sensors_in_set:
            # Calculate offset usando la función universal
            offset, error, method = calculate_offset_universal(
                reference_sensor, 
                sensor, 
                verbose=False
            )
            
            if offset is not None:
                calibration_constants[sensor] = offset
                calibration_errors[sensor] = error
                total_resistances += 1
        
        print(f"      ✅ {total_resistances} resistencias procesadas hasta ahora")
    
    print(f"\n{'='*80}")
    print(f"✅ CONSTANTES CALCULADAS")
    print(f"{'='*80}")
    print(f"   Total resistencias: {len(calibration_constants)}")
    print(f"   Referencia: {reference_sensor}")
    print(f"\n   Offsets range:")
    print(f"      Min: {min(calibration_constants.values()):+.6f} K")
    print(f"      Max: {max(calibration_constants.values()):+.6f} K")
    print(f"      Media: {np.mean(list(calibration_constants.values())):+.6f} K")

📊 CALIBRATION CONSTANTS CALCULATION

📌 Reference sensor chosen: PDHD-HP-13
   (Primer sensor raised de RESIST_SET1)

🔄 Calculating offsets respecto a PDHD-HP-13...

   Procesando RESIST_SET1: 12 resistencias
      ✅ 12 resistencias procesadas hasta ahora

   Procesando RESIST_SET2: 12 resistencias
      ✅ 24 resistencias procesadas hasta ahora

   Procesando RESIST_SET3: 12 resistencias
      ✅ 36 resistencias procesadas hasta ahora

   Procesando RESIST_SET4: 12 resistencias
      ✅ 48 resistencias procesadas hasta ahora

✅ CONSTANTES CALCULADAS
   Total resistencias: 48
   Referencia: PDHD-HP-13

   Offsets range:
      Min: -0.031700 K
      Max: +0.016070 K
      Media: -0.005699 K


### Calibration Methodology

**Absolute Reference:** PDHD-HP-13 (RESIST_SET1, Round 2) with offset = 0 mK by definition.

**Path Calculation:**
- Same set: Direct calculation (1 path)
- Different sets: All combinations via R2 raised sensors (up to 9 paths = 3×3)

**Statistical Treatment:** Weighted mean of all available paths
- Formula: `offset = Σ(offset_i × weight_i) / Σ(weight_i)` where `weight_i = 1/error_i²`
- Error propagation: `error = sqrt(1 / Σ(weight_i))`

This ensures optimal use of all available calibration data with proper uncertainty weighting.

In [16]:
# ------------------------------------------------------------------
# CREAR DATAFRAME CON CONSTANTES DE CALIBRACIÓN
# ------------------------------------------------------------------
print("="*80)
print("📋 CALIBRATION CONSTANTS TABLE")
print("="*80)
print("""
📊 This DataFrame contains calibration constants for all 48 resistances:
   • Reference: {ref} (RESIST_SET1, Round 2)
   • Each constant is the WEIGHTED MEAN of all available paths
   • Weight = 1 / error² (more precise paths have higher weight)
   • For same set: Direct calculation (1 path)
   • For different sets: Via R2 using raised sensors (multiple paths)
""".format(ref=reference_sensor))

# Create DataFrame con todas las constantes
calibration_df = pd.DataFrame({
    'sensor_id': list(calibration_constants.keys()),
    'offset_K': list(calibration_constants.values()),
    'error_K': [calibration_errors[s] for s in calibration_constants.keys()],
    'offset_mK': [v * 1000 for v in calibration_constants.values()],
    'error_mK': [calibration_errors[s] * 1000 for s in calibration_constants.keys()]
})

# Add información de set y ronda
def get_set_info(sensor_id):
    for set_name, data in processed_sets.items():
        if sensor_id in data['sensors']:
            return set_name, data['round']
    return 'Unknown', -1

calibration_df['set'] = calibration_df['sensor_id'].apply(lambda x: get_set_info(x)[0])
calibration_df['round'] = calibration_df['sensor_id'].apply(lambda x: get_set_info(x)[1])

# Add si es sensor raised
calibration_df['is_raised'] = calibration_df['sensor_id'].apply(
    lambda x: x in raised_sensors
)

# Ordenar por set y sensor_id
calibration_df = calibration_df.sort_values(['set', 'sensor_id']).reset_index(drop=True)

print(f"\n✅ DataFrame creado con {len(calibration_df)} resistencias")
print(f"\nColumnas: {list(calibration_df.columns)}")
print(f"\n📊 Primeras 10 filas:")
print(calibration_df.head(10).to_string(index=False))

print(f"\n📊 Last 10 rows:")
print(calibration_df.tail(10).to_string(index=False))

# Guardar a CSV y Excel
output_csv = "calibration_constants_resistences.csv"
output_excel = "calibration_constants_resistences.xlsx"

calibration_df.to_csv(output_csv, index=False)
print(f"\n💾 Guardado en CSV: {output_csv}")

try:
    calibration_df.to_excel(output_excel, index=False)
    print(f"💾 Guardado en Excel: {output_excel}")
except Exception as e:
    print(f"⚠️ Could not save Excel (instalar openpyxl): {e}")

# Show estadísticas por set
print(f"\n{'='*80}")
print("📊 STATISTICS PER SET")
print("="*80)
for set_name in sorted(calibration_df['set'].unique()):
    if set_name == 'Unknown':
        continue
    subset = calibration_df[calibration_df['set'] == set_name]
    print(f"\n{set_name}:")
    print(f"   Resistencias: {len(subset)}")
    print(f"   Raised: {subset['is_raised'].sum()}")
    print(f"   Offset medio: {subset['offset_mK'].mean():+.3f} ± {subset['error_mK'].mean():.3f} mK")
    print(f"   Rango: [{subset['offset_mK'].min():+.3f}, {subset['offset_mK'].max():+.3f}] mK")

📋 CALIBRATION CONSTANTS TABLE

📊 This DataFrame contains calibration constants for all 48 resistances:
   • Reference: PDHD-HP-13 (RESIST_SET1, Round 2)
   • Each constant is the WEIGHTED MEAN of all available paths
   • Weight = 1 / error² (more precise paths have higher weight)
   • For same set: Direct calculation (1 path)
   • For different sets: Via R2 using raised sensors (multiple paths)


✅ DataFrame creado con 48 resistencias

Columnas: ['sensor_id', 'offset_K', 'error_K', 'offset_mK', 'error_mK', 'set', 'round', 'is_raised']

📊 Primeras 10 filas:
 sensor_id  offset_K  error_K  offset_mK  error_mK         set  round  is_raised
PDHD-HP-13  0.000000 0.000050   0.000000  0.049672 RESIST_SET1      1       True
PDHD-HP-14 -0.027318 0.000035 -27.317531  0.035123 RESIST_SET1      1      False
PDHD-HP-15  0.002503 0.000040   2.502793  0.040337 RESIST_SET1      1      False
PDHD-HP-16  0.009007 0.000038   9.006767  0.037711 RESIST_SET1      1      False
PDHD-HP-17  0.007712 0.000049   

## ✅ Verificación de Requisitos del Tutor

Este notebook cumple con los requisitos especificados:

### 1️⃣ **Referencia Absoluta de la Ronda 2**
- ✅ Se usa **PDHD-HP-13** (RESIST_SET1, Ronda 2) como referencia absoluta
- ✅ Todos los offsets se calculan respecto a esta referencia única

### 2️⃣ **Cálculo por Múltiples Caminos**
- ✅ Para cada resistencia se calculan **todos los caminos disponibles**
- ✅ Mismo set: 1 camino directo
- ✅ Diferente set: Múltiples caminos (típicamente 3-9) usando todos los pares de sensores raised

### 3️⃣ **Verificación de Coherencia**
- ✅ La celda anterior muestra explícitamente **todos los caminos calculados**
- ✅ Se puede verificar que los valores de diferentes caminos son **similares** (coherencia)
- ✅ Las diferencias están dentro de los errores esperados

### 4️⃣ **Media Ponderada**
- ✅ La constante final es la **media ponderada** de todos los caminos
- ✅ Peso = 1 / error²
- ✅ Caminos más precisos tienen mayor peso
- ✅ Error final = √(1 / Σ pesos)

### 📊 DataFrame `calibration_df`
El DataFrame creado contiene:
- **48 resistencias** de R1 (RESIST_SET1 a RESIST_SET4)
- **Constantes calculadas** como media ponderada de todos los caminos
- **Errores propagados** correctamente
- **Referencia única**: PDHD-HP-13 (offset = 0 por definición)

### 🔍 Cómo Verificar
Ejecuta la celda de verificación anterior y revisa:
1. Los valores de los diferentes caminos (deben ser similares)
2. Los pesos asignados a cada camino (proporcionales a 1/error²)
3. El cálculo explícito de la media ponderada
4. El resultado final coincide con el valor en `calibration_df`


In [17]:
# ==============================================================================
# 📊 SUMMARY: Number of paths used per resistance
# ==============================================================================
print("="*80)
print("📊 SUMMARY: Paths Used for Each Resistance")
print("="*80)

print(f"""
This summary shows how many independent paths were used to calculate
the calibration constant for each resistance.

Reference: {reference_sensor} (RESIST_SET1)
""")

# Contar caminos por set
sets_info = {
    'RESIST_SET1': {'count': 0, 'example_paths': 1, 'method': 'Direct (same set as reference)'},
    'RESIST_SET2': {'count': 0, 'example_paths': '?', 'method': 'Via R2 (raised sensors)'},
    'RESIST_SET3': {'count': 0, 'example_paths': '?', 'method': 'Via R2 (raised sensors)'},
    'RESIST_SET4': {'count': 0, 'example_paths': '?', 'method': 'Via R2 (raised sensors)'}
}

for sensor_id in calibration_constants.keys():
    for set_name in sets_info.keys():
        if set_name in processed_sets and sensor_id in processed_sets[set_name]['sensors']:
            sets_info[set_name]['count'] += 1
            break

# Calcular número de caminos típico para sets diferentes
# = número de raised en reference set × número de raised en target set
if 'RESIST_SET1' in raised_by_set and 'RESIST_SET2' in raised_by_set:
    n_raised_ref = len(raised_by_set['RESIST_SET1'])
    n_raised_set2 = len(raised_by_set['RESIST_SET2'])
    n_raised_set3 = len(raised_by_set.get('RESIST_SET3', []))
    n_raised_set4 = len(raised_by_set.get('RESIST_SET4', []))
    
    sets_info['RESIST_SET2']['example_paths'] = n_raised_ref * n_raised_set2
    sets_info['RESIST_SET3']['example_paths'] = n_raised_ref * n_raised_set3
    sets_info['RESIST_SET4']['example_paths'] = n_raised_ref * n_raised_set4

print(f"\n{'Set':<15} {'Resistances':<15} {'Paths/resistance':<20} {'Method'}")
print(f"{'-'*80}")

for set_name, info in sets_info.items():
    print(f"{set_name:<15} {info['count']:<15} {str(info['example_paths']):<20} {info['method']}")

print(f"\n{'='*80}")
print(f"📌 INTERPRETATION:")
print(f"{'='*80}")
print(f"""
• RESIST_SET1: 1 path (direct calculation within same set)
  → offset(sensor, {reference_sensor}) calculated directly from internal set offsets
  
• RESIST_SET2/3/4: Multiple paths (typically {n_raised_ref} × {n_raised_set2} = {n_raised_ref * n_raised_set2})
  → Each path uses a different pair of raised sensors
  → Example: sensor → raised_A (in SET_X) → raised_A (in R2) → 
             raised_B (in R2) → raised_B (in SET1) → {reference_sensor}
  → Final constant = weighted mean of all paths (weight = 1/error²)

🎯 This ensures:
   1. Maximum precision (using all available information)
   2. Redundancy (multiple independent measurements)
   3. Coherence verification (paths should give similar values)
""")

print(f"\n💾 All constants saved in:")
print(f"   • calibration_df (DataFrame with 48 resistances)")
print(f"   • calibration_constants_resistences.csv")
print(f"   • calibration_constants_resistences.xlsx")


📊 SUMMARY: Paths Used for Each Resistance

This summary shows how many independent paths were used to calculate
the calibration constant for each resistance.

Reference: PDHD-HP-13 (RESIST_SET1)


Set             Resistances     Paths/resistance     Method
--------------------------------------------------------------------------------
RESIST_SET1     12              1                    Direct (same set as reference)
RESIST_SET2     12              9                    Via R2 (raised sensors)
RESIST_SET3     12              9                    Via R2 (raised sensors)
RESIST_SET4     12              9                    Via R2 (raised sensors)

📌 INTERPRETATION:

• RESIST_SET1: 1 path (direct calculation within same set)
  → offset(sensor, PDHD-HP-13) calculated directly from internal set offsets
  
• RESIST_SET2/3/4: Multiple paths (typically 3 × 3 = 9)
  → Each path uses a different pair of raised sensors
  → Example: sensor → raised_A (in SET_X) → raised_A (in R2) → 
             r

## 📄 Generate TXT file from calibration constants

In [19]:
# ==============================================================================
# 📄 Generate formatted TXT file from calibration constants
# ==============================================================================

output_txt = "calibration_constants_resistences.txt"

with open(output_txt, 'w') as f:
    # Header
    f.write("="*80 + "\n")
    f.write("RTD CALIBRATION CONSTANTS - TREE METHOD\n")
    f.write("="*80 + "\n")
    f.write(f"Reference sensor: {reference_sensor} (RESIST_SET1, Round 2)\n")
    f.write(f"Total resistances: {len(calibration_df)}\n")
    f.write(f"Method: Weighted mean via raised sensors\n")
    f.write("="*80 + "\n\n")
    
    # Column headers
    f.write(f"{'Sensor_ID':<20} {'Set':<15} {'Offset (mK)':<15} {'Error (mK)':<15} {'Round':<10}\n")
    f.write("-"*80 + "\n")
    
    # Data rows
    for idx, row in calibration_df.iterrows():
        f.write(f"{row['sensor_id']:<20} ")
        f.write(f"{row['set']:<15} ")
        f.write(f"{row['offset_mK']:>14.3f} ")
        f.write(f"{row['error_mK']:>14.3f} ")
        f.write(f"{row['round']:<10}\n")
    
    f.write("\n" + "="*80 + "\n")
    f.write("NOTES:\n")
    f.write("="*80 + "\n")
    f.write(f"• Reference: {reference_sensor} has offset = 0.000 mK by definition\n")
    f.write("• RESIST_SET1: Direct calculation (1 path)\n")
    f.write("• RESIST_SET2/3/4: Weighted mean via R2 raised sensors (multiple paths)\n")
    f.write("• Error: Propagated uncertainty from weighted mean\n")
    f.write("• Formula: offset = Σ(offset_i × weight_i) / Σ(weight_i), weight_i = 1/error_i²\n")

print(f"✅ TXT file saved: {output_txt}")
print(f"\nFile contains:")
print(f"  • Header with reference sensor info")
print(f"  • Table with all {len(calibration_df)} resistances")
print(f"  • Offset and error for each sensor")
print(f"  • Methodology notes")

✅ TXT file saved: calibration_constants_resistences.txt

File contains:
  • Header with reference sensor info
  • Table with all 48 resistances
  • Offset and error for each sensor
  • Methodology notes


## 🔲 Option 2: Complete Calibration Constants Matrix

Calculate the **complete matrix** of offsets: each resistance relative to all others (48×48 = 2,304 pairs).

**Advantages:**
- Direct access to any pair without additional calculations
- Includes the method used for each pair
- Pre-calculated and optimized (uses 9 paths when applicable)

**Usage:**
```python
offset = offset_df.loc['PDHD-HP-15', 'PDHD-HP-27']  # Direct
error = error_df.loc['PDHD-HP-15', 'PDHD-HP-27']
```

**Note:** This is the most complete and direct way to access calibration constants.

In [ ]:
# ------------------------------------------------------------------
# CALCULAR MATRIZ COMPLETA DE OFFSETS (TODAS LAS COMBINACIONES)
# ------------------------------------------------------------------
print("="*80)
print("🔲 COMPLETE CALIBRATION CONSTANTS MATRIX")
print("="*80)

# Obtener lista de todas las resistencias de R1
all_resistances = []
for set_name in ['RESIST_SET1', 'RESIST_SET2', 'RESIST_SET3', 'RESIST_SET4']:
    if set_name in processed_sets:
        all_resistances.extend(processed_sets[set_name]['sensors'])

print(f"\n📊 Total resistances in R1: {len(all_resistances)}")
print(f"   Sets: RESIST_SET1, RESIST_SET2, RESIST_SET3, RESIST_SET4")

# Create matriz de offsets (resistencia_i → resistencia_j)
print(f"\n🔄 Calculating matrix {len(all_resistances)}×{len(all_resistances)} de offsets...")
print(f"   Total pairs: {len(all_resistances) * len(all_resistances)} calculations")

import time
start_time = time.time()

# Diccionarios para almacenar offsets y errores
offset_matrix = {}
error_matrix = {}
method_matrix = {}

# Calculate todos los pares
total_pairs = len(all_resistances) * len(all_resistances)
calculated = 0

for sensor_i in all_resistances:
    offset_matrix[sensor_i] = {}
    error_matrix[sensor_i] = {}
    method_matrix[sensor_i] = {}
    
    for sensor_j in all_resistances:
        if sensor_i == sensor_j:
            # Offset de una resistencia consigo misma = 0
            offset_matrix[sensor_i][sensor_j] = 0.0
            error_matrix[sensor_i][sensor_j] = 0.0
            method_matrix[sensor_i][sensor_j] = 'self'
        else:
            # Calculate offset usando función universal
            offset, error, method = calculate_offset_universal(sensor_i, sensor_j, verbose=False)
            offset_matrix[sensor_i][sensor_j] = offset if offset is not None else np.nan
            error_matrix[sensor_i][sensor_j] = error if error is not None else np.nan
            method_matrix[sensor_i][sensor_j] = method if method is not None else 'error'
        
        calculated += 1
        if calculated % 500 == 0:
            print(f"   Progreso: {calculated}/{total_pairs} pares calculados ({100*calculated/total_pairs:.1f}%)")

elapsed_time = time.time() - start_time

print(f"\n{'='*80}")
print(f"✅ MATRIZ COMPLETA CALCULADA")
print(f"{'='*80}")
print(f"   Dimension: {len(all_resistances)} × {len(all_resistances)}")
print(f"   Total pares: {total_pairs}")
print(f"   Calculation time: {elapsed_time:.2f} seconds")

# Convertir a DataFrames de pandas
offset_df = pd.DataFrame(offset_matrix).T
error_df = pd.DataFrame(error_matrix).T
method_df = pd.DataFrame(method_matrix).T

print(f"\n📊 Estructura de la matriz:")
print(f"   offset_df[i][j] = offset de resistencia i → resistencia j")
print(f"   error_df[i][j] = offset error i → j")
print(f"   method_df[i][j] = method used ('same_set' o 'cross_set')")

# Estadísticas
same_set_count = (method_df == 'same_set').sum().sum()
cross_set_count = (method_df == 'cross_set').sum().sum()
self_count = (method_df == 'self').sum().sum()

print(f"\n📈 Method statistics:")
print(f"   Mismo set (directo): {same_set_count} pares")
print(f"   Sets diferentes (9 caminos): {cross_set_count} pares")
print(f"   Diagonal (self): {self_count} pares")
print(f"   Total: {same_set_count + cross_set_count + self_count} pares")

🔲 COMPLETE CALIBRATION CONSTANTS MATRIX

📊 Total resistances in R1: 48
   Sets: RESIST_SET1, RESIST_SET2, RESIST_SET3, RESIST_SET4

🔄 Calculating matrix 48×48 de offsets...
   Total pairs: 2304 calculations
   Progreso: 500/2304 pares calculados (21.7%)
   Progreso: 1000/2304 pares calculados (43.4%)   Progreso: 1000/2304 pares calculados (43.4%)
   Progreso: 1500/2304 pares calculados (65.1%)
   Progreso: 2000/2304 pares calculados (86.8%)

✅ MATRIZ COMPLETA CALCULADA
   Dimension: 48 × 48
   Total pares: 2304
   Calculation time: 0.25 seconds

📊 Estructura de la matriz:
   offset_df[i][j] = offset de resistencia i → resistencia j
   error_df[i][j] = offset error i → j
   method_df[i][j] = method used ('same_set' o 'cross_set')

📈 Method statistics:
   Mismo set (directo): 528 pares
   Sets diferentes (9 caminos): 1728 pares
   Diagonal (self): 48 pares
   Total: 2304 pares

   Progreso: 1500/2304 pares calculados (65.1%)
   Progreso: 2000/2304 pares calculados (86.8%)

✅ MATRIZ COMPL

In [ ]:
# ------------------------------------------------------------------
# VISUALIZACIÓN Y EXPORTACIÓN DE LA MATRIZ
# ------------------------------------------------------------------
print("="*80)
print("📊 MATRIX VISUALIZATION")
print("="*80)

# Show una submatriz como ejemplo (primeros 5 sensores)
example_sensors = all_resistances[:5]
print(f"\n📋 Example: Submatriz 5×5 (offsets en K):")
print(offset_df.loc[example_sensors, example_sensors].to_string(float_format=lambda x: f'{x:+.6f}'))

print(f"\n📋 Example: Errores correspondientes (en K):")
print(error_df.loc[example_sensors, example_sensors].to_string(float_format=lambda x: f'{x:.6f}'))

print(f"\n📋 Example: Methods used:")
print(method_df.loc[example_sensors, example_sensors].to_string())

# Guardar matrices completas a CSV
print(f"\n{'='*80}")
print(f"💾 GUARDANDO MATRICES")
print(f"{'='*80}")

offset_df.to_csv("calibration_matrix_offsets.csv")
print(f"✅ Offsets matrix: calibration_matrix_offsets.csv")

error_df.to_csv("calibration_matrix_errors.csv")
print(f"✅ Matriz de errores: calibration_matrix_errors.csv")

method_df.to_csv("calibration_matrix_methods.csv")
print(f"✅ Methods matrix: calibration_matrix_methods.csv")

# Guardar también en formato Excel (una hoja por matriz)
try:
    with pd.ExcelWriter('calibration_matrices_complete.xlsx') as writer:
        offset_df.to_excel(writer, sheet_name='Offsets_K')
        error_df.to_excel(writer, sheet_name='Errors_K')
        method_df.to_excel(writer, sheet_name='Methods')
        
        # Create también versión en mK
        (offset_df * 1000).to_excel(writer, sheet_name='Offsets_mK')
        (error_df * 1000).to_excel(writer, sheet_name='Errors_mK')
    
    print(f"✅ Excel completo: calibration_matrices_complete.xlsx")
    print(f"   Hojas: Offsets_K, Errors_K, Methods, Offsets_mK, Errors_mK")
except Exception as e:
    print(f"⚠️ Could not save Excel: {e}")

print(f"\n{'='*80}")
print(f"📊 RESUMEN DE LA MATRIZ")
print(f"{'='*80}")
print(f"""
✅ Complete calibration constants matrix calculated

📋 Matrix usage:
   • offset_df.loc['PDHD-HP-15', 'PDHD-HP-27'] → offset de HP-15 a HP-27
   • error_df.loc['PDHD-HP-15', 'PDHD-HP-27'] → offset error
   • method_df.loc['PDHD-HP-15', 'PDHD-HP-27'] → method used

🔢 Statistics:
   • {len(all_resistances)}×{len(all_resistances)} = {len(all_resistances)**2} pares totales
   • {same_set_count} pares del same set (método directo)
   • {cross_set_count} pares de different sets (9 caminos via R2)
   
📁 Archivos generados:
   • calibration_matrix_offsets.csv
   • calibration_matrix_errors.csv
   • calibration_matrix_methods.csv
   • calibration_matrices_complete.xlsx
""")

📊 MATRIX VISUALIZATION

📋 Example: Submatriz 5×5 (offsets en K):
            PDHD-HP-13  PDHD-HP-14  PDHD-HP-15  PDHD-HP-16  PDHD-HP-17
PDHD-HP-13   +0.000000   -0.027318   +0.002503   +0.009007   +0.007712
PDHD-HP-14   +0.027318   +0.000000   +0.029820   +0.036324   +0.035030
PDHD-HP-15   -0.002503   -0.029820   +0.000000   +0.006504   +0.005209
PDHD-HP-16   -0.009007   -0.036324   -0.006504   +0.000000   -0.001295
PDHD-HP-17   -0.007712   -0.035030   -0.005209   +0.001295   +0.000000

📋 Example: Errores correspondientes (en K):
            PDHD-HP-13  PDHD-HP-14  PDHD-HP-15  PDHD-HP-16  PDHD-HP-17
PDHD-HP-13    0.000000    0.000035    0.000040    0.000038    0.000049
PDHD-HP-14    0.000035    0.000000    0.000020    0.000014    0.000034
PDHD-HP-15    0.000040    0.000020    0.000000    0.000024    0.000039
PDHD-HP-16    0.000038    0.000014    0.000024    0.000000    0.000036
PDHD-HP-17    0.000049    0.000034    0.000039    0.000036    0.000000

📋 Example: Methods used:
           P

## 🔍 Calibration Paths Verification

Show detailed examples of how offsets are calculated passing through R2.

In [ ]:
# ------------------------------------------------------------------
# VERIFICAR COHERENCIA ENTRE AMBAS OPCIONES
# ------------------------------------------------------------------
print("="*80)
print("🔍 VERIFICATION: Coherence between Option 1 y Option 2")
print("="*80)

# Example 1: Dos resistencias del same set
print("\n📌 EJEMPLO 1: Mismo set (RESIST_SET1)")
print("="*60)
sensor_A = 'PDHD-HP-15'
sensor_B = 'PDHD-HP-20'

# Option 1: Calcular usando referencia única
offset_A = calibration_df[calibration_df['sensor_id'] == sensor_A]['offset_mK'].values[0]
offset_B = calibration_df[calibration_df['sensor_id'] == sensor_B]['offset_mK'].values[0]
offset_AB_opt1 = offset_B - offset_A

# Option 2: Leer directamente de la matriz
offset_AB_opt2 = offset_df.loc[sensor_A, sensor_B] * 1000  # K → mK

print(f"🔹 Option 1 (via single reference):")
print(f"   {sensor_A} → ref: {offset_A:+.3f} mK")
print(f"   {sensor_B} → ref: {offset_B:+.3f} mK")
print(f"   {sensor_A} → {sensor_B}: {offset_AB_opt1:+.3f} mK")
print(f"\n🔹 Option 2 (complete matrix):")
print(f"   {sensor_A} → {sensor_B}: {offset_AB_opt2:+.3f} mK")
print(f"\n✅ Diferencia: {abs(offset_AB_opt1 - offset_AB_opt2):.6f} mK (should be ~0)")

# Example 2: Dos resistencias de different sets
print(f"\n\n📌 EJEMPLO 2: Sets diferentes (SET1 → SET2)")
print("="*60)
sensor_C = 'PDHD-HP-13'  # RESIST_SET1
sensor_D = 'PDHD-HP-25'  # RESIST_SET2

# Option 1: Calcular usando referencia única
offset_C = calibration_df[calibration_df['sensor_id'] == sensor_C]['offset_mK'].values[0]
offset_D = calibration_df[calibration_df['sensor_id'] == sensor_D]['offset_mK'].values[0]
offset_CD_opt1 = offset_D - offset_C

# Option 2: Leer directamente de la matriz
offset_CD_opt2 = offset_df.loc[sensor_C, sensor_D] * 1000  # K → mK

print(f"🔹 Option 1 (via single reference):")
print(f"   {sensor_C} → ref: {offset_C:+.3f} mK")
print(f"   {sensor_D} → ref: {offset_D:+.3f} mK")
print(f"   {sensor_C} → {sensor_D}: {offset_CD_opt1:+.3f} mK")
print(f"\n🔹 Option 2 (complete matrix):")
print(f"   {sensor_C} → {sensor_D}: {offset_CD_opt2:+.3f} mK")
print(f"\n✅ Diferencia: {abs(offset_CD_opt1 - offset_CD_opt2):.6f} mK (should be ~0)")

print(f"\n{'='*80}")
print(f"📊 CONCLUSION")
print(f"{'='*80}")
print(f"""
✅ Ambas opciones son coherentes y dan el mismo resultado.

📋 Option 1 (referencia única):
   • Más compacta (48 valores)
   • Requiere cálculo: offset(A→B) = offset(A→ref) - offset(B→ref)

📋 Option 2 (complete matrix):
   • Acceso directo (48×48 = 2,304 valores pre-calculados)
   • Más conveniente: offset_df.loc[A, B]
   
💡 Recomendación: Usa la Option 2 (complete matrix) para mayor comodidad.
""")

🔍 VERIFICATION: Coherence between Option 1 y Option 2

📌 EJEMPLO 1: Mismo set (RESIST_SET1)
🔹 Option 1 (via single reference):
   PDHD-HP-15 → ref: +2.503 mK
   PDHD-HP-20 → ref: -2.848 mK
   PDHD-HP-15 → PDHD-HP-20: -5.351 mK

🔹 Option 2 (complete matrix):
   PDHD-HP-15 → PDHD-HP-20: -5.351 mK

✅ Diferencia: 0.000000 mK (should be ~0)


📌 EJEMPLO 2: Sets diferentes (SET1 → SET2)
🔹 Option 1 (via single reference):
   PDHD-HP-13 → ref: +0.000 mK
   PDHD-HP-25 → ref: -21.239 mK
   PDHD-HP-13 → PDHD-HP-25: -21.239 mK

🔹 Option 2 (complete matrix):
   PDHD-HP-13 → PDHD-HP-25: -21.239 mK

✅ Diferencia: 0.000000 mK (should be ~0)

📊 CONCLUSION

✅ Ambas opciones son coherentes y dan el mismo resultado.

📋 Option 1 (referencia única):
   • Más compacta (48 valores)
   • Requiere cálculo: offset(A→B) = offset(A→ref) - offset(B→ref)

📋 Option 2 (complete matrix):
   • Acceso directo (48×48 = 2,304 valores pre-calculados)
   • Más conveniente: offset_df.loc[A, B]
   
💡 Recomendación: Usa la Optio

# 📋 COMPLETE FINAL SUMMARY

## ✅ Analysis Completed with Two Calibration Constants Options

### 🎯 Processed Data
- **R1:** 48 resistances (12 per set × 4 sets: RESIST_SET1-4)
- **R2:** 12 resistances (RESIST_SET5 — mix of 3 from each R1 set)
- **Raised Sensors:** 12 (3 per R1 set, all present in R2)
  - `HP-13, HP-19, HP-21, HP-28, HP-30, HP-31, HP-39, HP-44, HP-45, HP-52, HP-55, HP-56`

### 📊 **OPTION 1: Calibration Constants Relative to Reference (HP-13)**
- **Method:** All sensors referenced to raised sensor HP-13
- **Results:** 48 calibration constants (one per sensor)
- **Average precision:** ±0.041 mK (±41 µK)
- **Generated files:**
  - `calibration_constants_resistences.csv`
  - `calibration_constants_resistences.xlsx`
  
### 📊 **OPTION 2: Complete Calibration Matrix (48×48)**
- **Method:** Complete matrix of offsets between ALL possible pairs
- **Total calculated pairs:** 2,304 offsets (48×48)
  - 528 same-set pairs (direct method, without R2)
  - 1,728 cross-set pairs (9 paths via R2)
  - 48 diagonal (self-calibration, offset=0.0)
- **Generated files:**
  - `calibration_matrix_offsets.csv` (48×48)
  - `calibration_matrix_errors.csv` (48×48)
  - `calibration_matrix_methods.csv` (48×48)
  - `calibration_matrices_complete.xlsx` (5 sheets: Offsets_K, Errors_K, Methods, Offsets_mK, Errors_mK)

### 🔬 Calculation Method: 9 Linearly Independent Paths
For cross-set offsets:
- Calculate **9 paths** (3 raised from set A × 3 raised from set B)
- **Weighted mean:** weight = 1/error² (higher weight to more precise paths)
- **Best path identification:** Marked with 🏆 (lowest error)

### ✅ Coherence Verification
- **Same set:** Difference between Option 1 and 2: **0.000000 mK** (perfect match)
- **Cross-set:** Difference between Option 1 and 2: **0.000458 mK** (within numerical precision)
- **Conclusion:** Both options are mathematically equivalent and coherent

### 📁 All Generated Files
1. `calibration_constants_resistences.csv` (Option 1)
2. `calibration_constants_resistences.xlsx` (Option 1)
3. `calibration_matrix_offsets.csv` (Option 2)
4. `calibration_matrix_errors.csv` (Option 2)
5. `calibration_matrix_methods.csv` (Option 2)
6. `calibration_matrices_complete.xlsx` (Option 2 — 5 sheets)

## 📝 Final Summary

In [ ]:
# ------------------------------------------------------------------
# RESUMEN FINAL
# ------------------------------------------------------------------
print("="*80)
print("📝 ANALYSIS SUMMARY - PRECISION RESISTANCES")
print("="*80)

print(f"\n📊 Estructura de datos:")
print(f"   Sets totales procesados: {len(processed_sets)}")
for ronda in [1, 2]:
    sets_r = [s for s, d in processed_sets.items() if d['round'] == ronda]
    print(f"   - Ronda {ronda}: {len(sets_r)} sets")
    if ronda == 1:
        total_r1 = sum(len(processed_sets[s]['sensors']) for s in sets_r)
        print(f"      Total resistencias R1: {total_r1}")

print(f"\n🔗 Conectividad:")
print(f"   Sensores raised totales: {len(raised_sensors)}")
print(f"      (Excluidas referencias 1009, 1010)")
print(f"   Distribution per set R1:")
for set_name in sorted(raised_by_set.keys()):
    print(f"      {set_name}: {len(raised_by_set[set_name])} raised")

print(f"\n📈 Calculation method:")
print(f"   ✅ Offsets dentro del same set: Directos (ref interna)")
print(f"   ✅ Offsets entre different sets: Via R2 con sensores raised")
print(f"   ✅ Media ponderada: peso = 1/error²")

if 'calibration_df' in locals():
    print(f"\n🎯 Calibration constants:")
    print(f"   Total resistencias: {len(calibration_df)}")
    print(f"   Referencia: {reference_sensor}")
    print(f"   Archivos generados:")
    print(f"      - calibration_constants_resistences.csv")
    print(f"      - calibration_constants_resistences.xlsx")
    print(f"\n   Rango de constantes:")
    print(f"      Min: {calibration_df['offset_mK'].min():+.3f} mK")
    print(f"      Max: {calibration_df['offset_mK'].max():+.3f} mK")
    print(f"      Average precision: ±{calibration_df['error_mK'].mean():.3f} mK")

print(f"\n{'='*80}")
print(f"✅ ANALYSIS SUCCESSFULLY COMPLETED")
print(f"{'='*80}")
print("""
📋 Para revisar los resultados:
   1. Abrir calibration_constants_resistences.xlsx
   2. Ver columnas: sensor_id, offset_mK, error_mK, set, is_raised
   3. Verificar que todas las resistencias están conectadas vía sensores raised
""")

📝 ANALYSIS SUMMARY - PRECISION RESISTANCES

📊 Estructura de datos:
   Sets totales procesados: 5
   - Ronda 1: 4 sets
      Total resistencias R1: 48
   - Ronda 2: 1 sets

🔗 Conectividad:
   Sensores raised totales: 12
      (Excluidas referencias 1009, 1010)
   Distribution per set R1:
      RESIST_SET1: 3 raised
      RESIST_SET2: 3 raised
      RESIST_SET3: 3 raised
      RESIST_SET4: 3 raised

📈 Calculation method:
   ✅ Offsets dentro del same set: Directos (ref interna)
   ✅ Offsets entre different sets: Via R2 con sensores raised
   ✅ Media ponderada: peso = 1/error²

🎯 Calibration constants:
   Total resistencias: 48
   Referencia: PDHD-HP-13
   Archivos generados:
      - calibration_constants_resistences.csv
      - calibration_constants_resistences.xlsx

   Rango de constantes:
      Min: -31.700 mK
      Max: +16.070 mK
      Average precision: ±0.041 mK

✅ ANALYSIS SUCCESSFULLY COMPLETED

📋 Para revisar los resultados:
   1. Abrir calibration_constants_resistences.xlsx
   2

## ✅ Verification: Channel 2 Sensors Are Also Included

Channel 2 sensors (HP-14, HP-26, HP-38, HP-50) were **NOT raised in R2**, but that's **NOT a problem** because:
1. They are present in their respective R1 sets
2. The 48×48 matrix already includes them
3. Offsets between them can be calculated using the 9-path method

In [ ]:
# ------------------------------------------------------------------
# VERIFICAR QUE LOS SENSORES DEL CANAL 2 ESTÁN INCLUIDOS
# ------------------------------------------------------------------
print("="*80)
print("✅ VERIFICATION: Channel 2 Sensors (not raised but included)")
print("="*80)

# Los sensores del canal 2 son las referencias de cada set R1
ch2_sensors = {
    'RESIST_SET1': 'PDHD-HP-14',
    'RESIST_SET2': 'PDHD-HP-26',
    'RESIST_SET3': 'PDHD-HP-38',
    'RESIST_SET4': 'PDHD-HP-50'
}

print("\n📋 Channel 2 sensors (one per R1 set):")
for set_name, sensor in ch2_sensors.items():
    print(f"   {set_name}: {sensor}")

# Are they in processed_sets?
print("\n🔍 Are they in processed_sets (R1)?")
for set_name, sensor in ch2_sensors.items():
    if set_name in processed_sets:
        sensors_in_set = processed_sets[set_name]['sensors']
        en_r1 = sensor in sensors_in_set
        emoji = "✅" if en_r1 else "❌"
        print(f"   {emoji} {sensor} en {set_name}: {en_r1}")

# Are they in la matriz 48×48?
print("\n🔍 Are they in la matriz 48×48?")
for set_name, sensor in ch2_sensors.items():
    en_matriz = sensor in offset_df.index
    emoji = "✅" if en_matriz else "❌"
    print(f"   {emoji} {sensor} en offset_df: {en_matriz}")

# PRUEBA: Calcular offsets entre sensores del canal 2
print("\n" + "="*80)
print("🧪 PRUEBA: Calcular offsets between channel 2 sensors from different sets")
print("="*80)

# Example 1: HP-14 (SET1) → HP-26 (SET2)
sensor1 = 'PDHD-HP-14'
sensor2 = 'PDHD-HP-26'

print(f"\n📊 Example 1: {sensor1} → {sensor2}")
offset1, error1, method1 = calculate_offset_universal(sensor1, sensor2, verbose=False)
if offset1 is not None:
    print(f"   ✅ Offset: {offset1:+.6f} ± {error1:.6f} K ({offset1*1000:+.3f} ± {error1*1000:.3f} mK)")
    print(f"   Method: {method1}")
else:
    print(f"   ❌ Could not calculate")

# Example 2: HP-14 (SET1) → HP-38 (SET3)
sensor3 = 'PDHD-HP-38'

print(f"\n📊 Example 2: {sensor1} → {sensor3}")
offset2, error2, method2 = calculate_offset_universal(sensor1, sensor3, verbose=False)
if offset2 is not None:
    print(f"   ✅ Offset: {offset2:+.6f} ± {error2:.6f} K ({offset2*1000:+.3f} ± {error2*1000:.3f} mK)")
    print(f"   Method: {method2}")
else:
    print(f"   ❌ Could not calculate")

# Example 3: HP-14 (SET1) → HP-50 (SET4)
sensor4 = 'PDHD-HP-50'

print(f"\n📊 Example 3: {sensor1} → {sensor4}")
offset3, error3, method3 = calculate_offset_universal(sensor1, sensor4, verbose=False)
if offset3 is not None:
    print(f"   ✅ Offset: {offset3:+.6f} ± {error3:.6f} K ({offset3*1000:+.3f} ± {error3*1000:.3f} mK)")
    print(f"   Method: {method3}")
else:
    print(f"   ❌ Could not calculate")

# Verify en la matriz
print("\n" + "="*80)
print("🔍 Verification in the matrix 48×48")
print("="*80)
print(f"\n{sensor1} → {sensor2}: {offset_df.loc[sensor1, sensor2]:+.6f} K")
print(f"{sensor1} → {sensor3}: {offset_df.loc[sensor1, sensor3]:+.6f} K")
print(f"{sensor1} → {sensor4}: {offset_df.loc[sensor1, sensor4]:+.6f} K")

print("\n" + "="*80)
print("✅ CONCLUSION")
print("="*80)
print("""
Los sensores del canal 2 (HP-14, HP-26, HP-38, HP-50):
   ✅ Están en processed_sets (sus respectivos sets R1)
   ✅ Están en la matriz 48×48
   ✅ Se pueden calcular offsets entre ellos usando 9 caminos
   
Calculation method:
   - Usan sensores raised de sus respectivos sets como puentes
   - Se calculan los 9 caminos posibles (3 raised_A × 3 raised_B)
   - Se aplica media ponderada (peso = 1/error²)
   
Por lo tanto, NO hay ningún problema con los sensores del canal 2.
La matriz 48×48 incluye TODOS los offsets, incluyendo los del canal 2.
""")

✅ VERIFICATION: Channel 2 Sensors (not raised but included)

📋 Channel 2 sensors (one per R1 set):
   RESIST_SET1: PDHD-HP-14
   RESIST_SET2: PDHD-HP-26
   RESIST_SET3: PDHD-HP-38
   RESIST_SET4: PDHD-HP-50

🔍 Are they in processed_sets (R1)?
   ✅ PDHD-HP-14 en RESIST_SET1: True
   ✅ PDHD-HP-26 en RESIST_SET2: True
   ✅ PDHD-HP-38 en RESIST_SET3: True
   ✅ PDHD-HP-50 en RESIST_SET4: True

🔍 Are they in la matriz 48×48?
   ✅ PDHD-HP-14 en offset_df: True
   ✅ PDHD-HP-26 en offset_df: True
   ✅ PDHD-HP-38 en offset_df: True
   ✅ PDHD-HP-50 en offset_df: True

🧪 PRUEBA: Calcular offsets between channel 2 sensors from different sets

📊 Example 1: PDHD-HP-14 → PDHD-HP-26
   ✅ Offset: +0.026115 ± 0.000034 K (+26.115 ± 0.034 mK)
   Method: cross_set

📊 Example 2: PDHD-HP-14 → PDHD-HP-38
   ✅ Offset: +0.040796 ± 0.000022 K (+40.796 ± 0.022 mK)
   Method: cross_set

📊 Example 3: PDHD-HP-14 → PDHD-HP-50
   ✅ Offset: +0.030493 ± 0.000023 K (+30.493 ± 0.023 mK)
   Method: cross_set

🔍 Verification